# QR-Phishing Detection

This notebook is a cleaned, publication-oriented version of the development notebook.

**Main paper story**

1. Data preparation and leakage control  
2. Lexical vs. QR-structural representation analysis  
3. Grouped model comparison  
4. Lightweight feature reduction  
5. Statistical / robustness validation  
6. Final held-out evaluation  
7. Interpretability and error analysis  
8. Deployment consistency  
9. External unseen-host validation  

> The original development notebook should be kept separately as the full research log.  
> This clean notebook intentionally omits abandoned routing/gating experiments, duplicate function definitions, and debugging cells.


## 0. Imports and configuration

In [ ]:
from pathlib import Path
import json
import pickle
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from urllib.parse import urlparse, urlsplit

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold
)
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = Path(".")


### Expected files in the same FYP folder

The clean notebook uses the artifacts already produced by the full development notebook.

Core files:

- `qr_full_feature_dataset_v2.csv`
- `baseline_all_feature_sets_grouped_cv.csv`
- `tuned_model_summary.csv`
- `phase12C_feature_budget_summary.csv`
- `phase13_final_significance_summary.csv`
- `phase14_robustness_summary.csv`
- `phase14B_feature_budget_selection.csv`
- `phase16_final_lightweight_efficiency_summary.csv`
- `phase17_final_heldout_test_summary.csv`
- `phase18_final10_permutation_importance.csv`
- `phase18_confidence_error_summary.csv`
- `FINAL_10feature_XGB_for_deployment.pkl`
- `FINAL_10feature_deployment_config.json`
- `balanced_urls.csv`
- Phase 21 external-validation CSVs

If a file is missing, run the corresponding section in the original development notebook once to regenerate it.


## 1. Load the cleaned primary dataset

In [ ]:
PRIMARY_DATA = DATA_DIR / "qr_full_feature_dataset_v2.csv"

df_all_features = pd.read_csv(PRIMARY_DATA)

print("Dataset shape:", df_all_features.shape)
print("\nClass distribution:")
print(df_all_features["label"].value_counts().sort_index())

print("\nUnique normalized URLs:", df_all_features["normalized_url"].nunique())
print("Unique hostnames:", df_all_features["hostname"].nunique())

df_all_features.head()


### Dataset audit summary used in the paper

- Original QR samples: **9,987**
- Successfully decoded URLs: **9,720**
- Final unique normalized URLs: **9,713**
- Benign: **4,900**
- Phishing: **4,813**
- Hostname-grouped train/test split is used to avoid hostname leakage.


## 2. Final feature definitions

In [ ]:
FINAL_10_FEATURES = [
    "num_subdomains",
    "suspicious_keyword_count",
    "num_digits",
    "path_length",
    "hostname_length",
    "url_length",
    "num_dots",
    "special_char_ratio",
    "num_hyphens",
    "query_length"
]

LEXICAL_14 = [
    "url_length",
    "hostname_length",
    "path_length",
    "query_length",
    "num_dots",
    "num_hyphens",
    "num_digits",
    "num_special_chars",
    "special_char_ratio",
    "num_subdomains",
    "has_query",
    "has_fragment",
    "contains_at",
    "contains_port",
]

STRUCTURAL_13 = [
    "qr_version",
    "black_ratio",
    "horizontal_transition_density",
    "vertical_transition_density",
    "overall_transition_density",
    "pattern_2x2_entropy",
    "mean_horizontal_run",
    "std_horizontal_run",
    "max_horizontal_run",
    "mean_vertical_run",
    "std_vertical_run",
    "max_vertical_run",
    # qr_matrix_size was removed as perfectly redundant with qr_version
]

print("Final deployment features:", len(FINAL_10_FEATURES))
print(FINAL_10_FEATURES)


## 3. Leakage-aware development / held-out split

In [ ]:
X = df_all_features[FINAL_10_FEATURES]
y = df_all_features["label"].astype(int)
groups = df_all_features["hostname"].astype(str)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

dev_idx, test_idx = next(gss.split(X, y, groups=groups))

X_dev = X.iloc[dev_idx].copy()
X_test = X.iloc[test_idx].copy()

y_dev = y.iloc[dev_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_dev = groups.iloc[dev_idx].copy()
groups_test = groups.iloc[test_idx].copy()

host_overlap = set(groups_dev).intersection(set(groups_test))

print("Development samples:", len(X_dev))
print("Held-out samples:", len(X_test))
print("Hostname overlap:", len(host_overlap))
print("Development positive rate:", round(y_dev.mean(), 4))
print("Held-out positive rate:", round(y_test.mean(), 4))


**Methodological rule for the paper:** all model selection and feature-selection decisions are described as development-set procedures. The held-out set is reserved for final evaluation only.

## 4. Statistical characterization and redundancy analysis

In [ ]:
def load_if_exists(filename):
    path = DATA_DIR / filename
    if path.exists():
        return pd.read_csv(path)
    print(f"Missing: {filename}")
    return None

continuous_stats = load_if_exists("continuous_statistical_results.csv")
binary_stats = load_if_exists("binary_statistical_results.csv")

if continuous_stats is not None:
    display(continuous_stats.head(15))

if binary_stats is not None:
    display(binary_stats)


Key statistical findings from the development work:

- `url_length` showed a large class effect.
- QR version / size were highly associated with URL length.
- `pattern_2x2_diversity` was constant and removed.
- Redundancy filtering removed perfectly or near-perfectly redundant representations such as `qr_matrix_size`.
- Statistical significance is used for characterization; predictive feature selection is evaluated separately.


## 5. Representation and baseline-model comparison

In [ ]:
baseline_results = load_if_exists(
    "baseline_all_feature_sets_grouped_cv.csv"
)

if baseline_results is not None:
    # show strongest configurations first
    sort_col = "mean_f1" if "mean_f1" in baseline_results.columns else baseline_results.columns[-1]
    display(
        baseline_results
        .sort_values(sort_col, ascending=False)
        .head(20)
    )


### Main representation result

The primary comparison showed:

- Lexical features provide the strongest predictive signal.
- QR structural features alone are informative but substantially weaker.
- Removing QR version reduces structural-only performance, supporting its role as a payload-capacity proxy.
- Adding intrinsic QR structure to lexical features did not improve the best lexical baseline.

This result is more important for the paper than listing every baseline model.


## 6. Hyperparameter tuning and final model selection

In [ ]:
tuned_models = load_if_exists("tuned_model_summary.csv")

if tuned_models is not None:
    display(tuned_models)


The development-stage winner was XGBoost on lexical features.  
For publication, avoid claiming novelty from XGBoost itself; the contribution is the evaluation and lightweight representation analysis.

## 7. Lightweight feature-budget analysis

In [ ]:
feature_budget = load_if_exists(
    "phase12C_feature_budget_summary.csv"
)

robust_budget = load_if_exists(
    "phase14B_feature_budget_selection.csv"
)

if feature_budget is not None:
    print("Feature-budget summary")
    display(feature_budget)

if robust_budget is not None:
    print("Robustness-guided budget selection")
    display(robust_budget)


### Final lightweight configuration

The robust selection procedure chose **10 lexical features**, reducing the feature count from 14 to 10 (**28.6% reduction**) while retaining almost all in-domain predictive performance.

For the conference paper, describe this as **feature-level lightweighting** rather than making knowledge distillation the central contribution.


## 8. Significance, robustness, and non-inferiority

In [ ]:
sig_summary = load_if_exists(
    "phase13_final_significance_summary.csv"
)

robust_summary = load_if_exists(
    "phase14_robustness_summary.csv"
)

efficiency_summary = load_if_exists(
    "phase16_final_lightweight_efficiency_summary.csv"
)

if sig_summary is not None:
    print("Significance summary")
    display(sig_summary)

if robust_summary is not None:
    print("Repeated grouped-CV robustness")
    display(robust_summary)

if efficiency_summary is not None:
    print("Final lightweight efficiency")
    display(efficiency_summary)


Interpretation rules for the paper:

- Do **not** describe the 5-fold Wilcoxon result as statistically significant when \(p=0.8125\) / when the selected comparison does not meet \(\alpha=0.05\).
- The final model is supported by robustness and non-inferiority evidence, not by a claim that it is strictly superior.
- The non-inferiority margin is \(\Delta=0.005\) absolute F1 and should be described as a **prespecified practical tolerance**.


## 9. Final held-out evaluation

In [ ]:
heldout = load_if_exists(
    "phase17_final_heldout_test_summary.csv"
)

comparison = load_if_exists(
    "phase17_full_vs_final10_test_comparison.csv"
)

if heldout is not None:
    display(heldout)

if comparison is not None:
    display(comparison)


Key held-out values from the completed experiment:

- Final 10-feature F1: **0.8811**
- Precision: **0.8963**
- Recall: **0.8664**
- AUROC: **0.9509**
- MCC: **0.7641**
- F1 difference vs. the previous full lexical model: approximately **−0.00414**
- Formal non-inferiority under the 0.005 margin: **passed**


## 10. Interpretability and error analysis

In [ ]:
perm_importance = load_if_exists(
    "phase18_final10_permutation_importance.csv"
)

error_summary = load_if_exists(
    "phase18_confidence_error_summary.csv"
)

if perm_importance is not None:
    display(perm_importance)

if error_summary is not None:
    display(error_summary)


Important interpretation:

Permutation importance is the **absolute drop in the scoring metric after permutation**.  
For example, an importance of 0.13 means an F1 decrease of approximately 0.13, not “13% relative importance”.


## 11. Final deployment feature extractor

In [ ]:
%pip install tldextract

In [ ]:
import tldextract

SUSPICIOUS_KEYWORDS = [
    "login", "verify", "verification", "account", "secure",
    "update", "password", "signin", "confirm", "payment",
    "bank", "billing", "support"
]

domain_extractor = tldextract.TLDExtract(suffix_list_urls=None)


def ensure_scheme(url):
    url = str(url).strip()
    if not url.lower().startswith(("http://", "https://")):
        return "http://" + url
    return url


def get_registered_domain(hostname):
    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return ""

    # Compatibility fix required to exactly reproduce the
    # training-time representation used by the original dataset.
    if hostname == "fed.us" or hostname.endswith(".fed.us"):
        return "fed.us"

    ext = domain_extractor(hostname)

    if not ext.domain:
        return hostname

    if ext.suffix:
        return ext.domain + "." + ext.suffix

    return ext.domain


def count_subdomains(hostname):
    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return 0

    registered_domain = get_registered_domain(hostname)

    hostname_parts = [p for p in hostname.split(".") if p]
    domain_parts = [p for p in registered_domain.split(".") if p]

    return max(len(hostname_parts) - len(domain_parts), 0)


def parse_deployment_url(raw_url):
    raw_url = str(raw_url).strip()
    parsed = urlsplit(ensure_scheme(raw_url))

    return {
        "raw_url": raw_url,
        "hostname": (parsed.hostname or "").lower(),
        "path": parsed.path or "",
        "query": parsed.query or "",
        "fragment": parsed.fragment or ""
    }


def extract_deployment_features(raw_url):
    p = parse_deployment_url(raw_url)

    raw_url = p["raw_url"]
    lower_url = raw_url.lower()
    hostname = p["hostname"]
    path = p["path"]
    query = p["query"]

    url_length = len(raw_url)
    num_special_chars = sum(not ch.isalnum() for ch in raw_url)

    return {
        "num_subdomains": count_subdomains(hostname),
        "suspicious_keyword_count": sum(
            keyword in lower_url
            for keyword in SUSPICIOUS_KEYWORDS
        ),
        "num_digits": sum(ch.isdigit() for ch in raw_url),
        "path_length": len(path),
        "hostname_length": len(hostname),
        "url_length": url_length,
        "num_dots": raw_url.count("."),
        "special_char_ratio": (
            num_special_chars / max(url_length, 1)
        ),
        "num_hyphens": raw_url.count("-"),
        "query_length": len(query)
    }


print(get_registered_domain("www.fpl.fs.fed.us"))
print(count_subdomains("www.fpl.fs.fed.us"))


## 12. Deployment consistency audit

In [ ]:
training_feature_data = df_all_features.copy()

consistency_records = []

for i, row in training_feature_data.reset_index(drop=True).iterrows():
    new_features = extract_deployment_features(row["url"])

    for feature in FINAL_10_FEATURES:
        old_value = float(row[feature])
        new_value = float(new_features[feature])

        consistency_records.append({
            "sample": i,
            "feature": feature,
            "old_value": old_value,
            "new_value": new_value,
            "difference": old_value - new_value
        })

deployment_consistency_df = pd.DataFrame(consistency_records)

mismatch_df = deployment_consistency_df[
    deployment_consistency_df["difference"].abs() > 1e-12
].copy()

print("Samples checked:", len(training_feature_data))
print("Feature values checked:", len(deployment_consistency_df))
print(
    "Maximum absolute difference:",
    deployment_consistency_df["difference"].abs().max()
)
print("Mismatch count:", len(mismatch_df))


Expected final audit result:

- Samples checked: **9,713**
- Feature values checked: **97,130**
- Mismatch count: **0**

This verifies that deployment-time feature extraction reproduces the training-time representation.


## 13. Load the frozen final deployment model

In [ ]:
%pip install xgboost

In [ ]:
MODEL_PATH = DATA_DIR / "FINAL_10feature_XGB_for_deployment.pkl"

final_deployment_model = joblib.load(MODEL_PATH)

print(final_deployment_model)


## 14. URL prediction helper

In [ ]:
def predict_url(url, threshold=0.5):
    features = extract_deployment_features(url)

    X_one = pd.DataFrame([features])[FINAL_10_FEATURES]

    prob = float(
        final_deployment_model.predict_proba(X_one)[0, 1]
    )

    pred = int(prob >= threshold)

    return {
        "url": url,
        "prediction": pred,
        "label": "PHISHING" if pred == 1 else "BENIGN",
        "phishing_probability": prob,
        "threshold": threshold,
        "features": features
    }


predict_url("https://www.google.com/")


The Google example is intentionally useful as an **error-analysis case**.  
The model's high phishing score is explained by a dataset-specific short-path pattern, not by a deployment bug. This should be discussed as a limitation rather than hard-coded away.


## 15. Optional QR-image prototype

In [ ]:
%pip install opencv-python

In [ ]:
import cv2


def decode_qr_image(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        return {
            "success": False,
            "decoded_text": None,
            "message": "Image could not be loaded."
        }

    detector = cv2.QRCodeDetector()
    decoded_text, points, _ = detector.detectAndDecode(image)

    if not decoded_text:
        return {
            "success": False,
            "decoded_text": None,
            "message": "A decodable QR code could not be detected."
        }

    return {
        "success": True,
        "decoded_text": decoded_text,
        "message": "QR code decoded successfully."
    }


def analyze_qr_image(image_path, threshold=0.5):
    qr = decode_qr_image(image_path)

    if not qr["success"]:
        return qr

    result = predict_url(
        qr["decoded_text"],
        threshold=threshold
    )

    result.update({
        "success": True,
        "decoded_url": qr["decoded_text"]
    })

    return result


## 16. External unseen-host validation

In [ ]:
EXTERNAL_FILE = r"C:\Users\LENOVO\Desktop\FYP\archive\balanced_urls.csv"

external_df = pd.read_csv(EXTERNAL_FILE)

print("External dataset:", external_df.shape)
print(external_df["result"].value_counts().sort_index())

In [ ]:
def get_external_hostname(url):
    try:
        return (
            urlsplit(ensure_scheme(str(url))).hostname or ""
        ).lower()
    except Exception:
        return ""


external_df["hostname"] = external_df["url"].apply(
    get_external_hostname
)

original_urls = set(
    training_feature_data["url"].astype(str)
)

original_hostnames = set(
    training_feature_data["hostname"].astype(str).str.lower()
)

external_df["url_overlap"] = (
    external_df["url"].astype(str).isin(original_urls)
)

external_df["hostname_overlap"] = (
    external_df["hostname"].isin(original_hostnames)
)

external_unseen = external_df[
    (~external_df["hostname_overlap"])
    & (external_df["hostname"] != "")
].copy()

print("External total:", len(external_df))
print("Unseen-host samples:", len(external_unseen))
print(
    "Unseen-host rate:",
    len(external_unseen) / len(external_df)
)
print(external_unseen["result"].value_counts().sort_index())


Completed external audit result:

- Total external URLs: **632,508**
- Unseen-host URLs after overlap removal: **616,947**
- Unseen-host proportion: **97.54%**
- External model evaluation uses a balanced 20,000-sample subset.
- The final model is frozen: **no retraining and no threshold tuning** on the external dataset.

Before publication, confirm whether the external positive class is *phishing only* or a broader *malicious URL* category. If broader, describe the experiment as an **out-of-domain malicious-URL stress test** rather than pure external phishing validation.


## 17. External validation results

In [ ]:
external_compare = load_if_exists(
    "phase21_internal_external_comparison.csv"
)

external_ci = load_if_exists(
    "phase21_external_ci_summary.csv"
)

external_shift = load_if_exists(
    "phase21_external_feature_shift.csv"
)

overlap_summary = load_if_exists(
    "phase21_external_overlap_summary.csv"
)

if external_compare is not None:
    display(external_compare)

if external_ci is not None:
    display(external_ci)

if overlap_summary is not None:
    display(overlap_summary)

if external_shift is not None:
    display(external_shift)


### External-generalization finding

Completed 20,000-sample unseen-host evaluation:

- F1: **0.7111**, 95% bootstrap CI **[0.7040, 0.7181]**
- AUROC: **0.7448**, 95% CI **[0.7381, 0.7516]**
- MCC: **0.4166**, 95% CI **[0.4041, 0.4288]**

Compared with the internal held-out evaluation, this is a substantial but systematic cross-dataset performance drop.

The distribution-shift analysis shows differences across multiple lexical variables, including suspicious-keyword frequency, digit count, subdomain count, dot count, special-character ratio, hyphen count, and query length.


## 18. Short-path bias analysis

In [ ]:
short_path_bias = load_if_exists(
    "phase19_short_path_bias_analysis.csv"
)

if short_path_bias is not None:
    display(short_path_bias)


A key limitation discovered during error analysis:

- Among samples with normalized `path_length <= 1`, approximately **95%** are phishing in the primary dataset.
- This creates a dataset-specific shortcut that can cause benign root-path URLs to receive high phishing scores.
- Do not post-hoc hard-code known benign domains to fix this; report the issue as evidence of dataset dependence.


## 19. Conference-paper result summary

In [ ]:
paper_summary = pd.DataFrame([
    {
        "Setting": "Internal held-out (10 features)",
        "F1": 0.881087,
        "AUROC": 0.950949,
        "MCC": 0.764140,
        "Notes": "Final frozen lightweight model"
    },
    {
        "Setting": "External unseen-host",
        "F1": 0.711124,
        "AUROC": 0.744839,
        "MCC": 0.416582,
        "Notes": "20,000 balanced external samples"
    }
])

paper_summary


## 20. What is intentionally excluded from the conference-clean notebook

The following development experiments are **not part of the recommended ESANN main story**:

- Confidence-gated routing
- Benefit-aware routing
- Utility-aware routing
- Multiple obsolete knowledge-distillation variants
- Earlier duplicate deployment extractors
- Debugging cells for individual parser mismatches
- Package-installation cells
- Intermediate hard-coded model comparisons that were superseded

They remain available in the original development notebook for traceability.

### Recommended paper positioning

The central contribution should be framed as:

> **Leakage-aware representation analysis + robust lightweight feature reduction + cross-dataset generalization evaluation**

rather than as a novel XGBoost classifier.


# Experiment 1 — Raw-Pixel Baseline under Hostname-Disjoint Evaluation
## Goal
Test whether the original QR raw-pixel signal remains strong when hostname/entity overlap is removed.
This notebook evaluates the original 69×69 QR pixels using the two strongest classical baselines from Trad & Chehab:
- XGBoost
- LightGBM
Three evaluation settings are reported:
1. Random 5-fold stratified CV
2. Hostname-grouped 5-fold CV
3. Final 80/20 hostname-disjoint holdout
### The important comparison is:
Random split performance - Hostname-disjoint performance
If performance drops materially after hostname separation, this is evidence that conventional random splitting may overestimate generalization.

### Important mapping
Your original notebook preserves:
original_index -> qr_codes[original_index]
after decoding, URL normalization, and duplicate removal. This notebook reconstructs the same mapping directly and verifies that labels still match.

In [ ]:
%pip install opencv-python pandas numpy scikit-learn xgboost lightgbm matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import cv2
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from urllib.parse import urlparse, urlunparse

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    GroupShuffleSplit
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
np.random.seed(SEED)

print("Ready.")

In [ ]:
DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f)).astype(int)

print("QR shape:", qr_codes.shape)
print("Labels:", labels.shape)
print("Class distribution:", dict(zip(*np.unique(labels, return_counts=True))))

assert qr_codes.shape == (9987, 69, 69)
assert labels.shape == (9987,)

In [ ]:
CACHE = Path("experiment1_qr_mapping.csv")

if CACHE.exists():
    mapping_df = pd.read_csv(CACHE)
    print("Loaded cached mapping:", CACHE)
else:
    detector = cv2.QRCodeDetector()
    decoded_urls = []

    for i, qr in enumerate(qr_codes):

        if qr.max() <= 1:
            img = (qr * 255).astype(np.uint8)
        else:
            img = qr.astype(np.uint8)

        img = cv2.copyMakeBorder(
            img, 4, 4, 4, 4,
            cv2.BORDER_CONSTANT,
            value=255
        )

        img = cv2.resize(
            img,
            None,
            fx=10,
            fy=10,
            interpolation=cv2.INTER_NEAREST
        )

        data, points, _ = detector.detectAndDecode(img)
        decoded_urls.append(data if data else None)

        if (i + 1) % 1000 == 0:
            print(f"Decoded {i+1}/{len(qr_codes)}")

    mapping_df = pd.DataFrame({
        "original_index": np.arange(len(qr_codes)),
        "url": decoded_urls,
        "label": labels
    })

    mapping_df.to_csv(CACHE, index=False)
    print("Saved:", CACHE)

print("Successfully decoded:", mapping_df["url"].notna().sum())
print("Failed:", mapping_df["url"].isna().sum())

In [ ]:
def normalize_url(url):
    url = str(url).strip()

    parse_url = url
    if not url.lower().startswith(("http://", "https://")):
        parse_url = "http://" + url

    parsed = urlparse(parse_url)

    hostname = parsed.hostname.lower() if parsed.hostname else ""

    if hostname.startswith("www."):
        hostname = hostname[4:]

    try:
        port = parsed.port
    except ValueError:
        port = None

    netloc = hostname
    if port is not None:
        netloc += f":{port}"

    scheme = parsed.scheme.lower()

    return urlunparse((
        scheme,
        netloc,
        parsed.path,
        "",
        parsed.query,
        ""
    ))

def extract_hostname(url):
    parse_url = str(url).strip()

    if not parse_url.lower().startswith(("http://", "https://")):
        parse_url = "http://" + parse_url

    parsed = urlparse(parse_url)
    hostname = parsed.hostname

    if hostname is None:
        return None

    hostname = hostname.lower()

    if hostname.startswith("www."):
        hostname = hostname[4:]

    return hostname

clean_df = mapping_df.dropna(subset=["url"]).copy()

clean_df["normalized_url"] = clean_df["url"].apply(normalize_url)

clean_df = (
    clean_df
    .drop_duplicates(subset=["normalized_url"])
    .copy()
)

clean_df["hostname"] = clean_df["url"].apply(extract_hostname)

clean_df = clean_df.dropna(subset=["hostname"]).copy()

print("Final mapped samples:", len(clean_df))
print("Label distribution:")
print(clean_df["label"].value_counts())
print("Unique hostnames:", clean_df["hostname"].nunique())

# Expected from your main notebook: approximately 9,713 final unique URLs.

In [ ]:
original_indices = clean_df["original_index"].astype(int).to_numpy()

X_pixels = qr_codes[original_indices].reshape(len(clean_df), -1).astype(np.float32)
y = clean_df["label"].astype(int).to_numpy()
groups = clean_df["hostname"].astype(str).to_numpy()

mapped_labels = labels[original_indices]

print("Raw pixel matrix:", X_pixels.shape)
print("Mapping label mismatches:", np.sum(mapped_labels != y))

assert np.sum(mapped_labels != y) == 0
assert X_pixels.shape[1] == 69 * 69

print("Mapping verified successfully.")

In [ ]:
def make_model(name):
    if name == "XGBoost":
        return XGBClassifier(
            learning_rate=0.2,
            n_estimators=150,
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1
        )

    if name == "LightGBM":
        return LGBMClassifier(
            learning_rate=0.1,
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1
        )

    raise ValueError(name)

In [ ]:
def compute_metrics(y_true, prob):
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": roc_auc_score(y_true, prob),
        "mcc": matthews_corrcoef(y_true, pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [ ]:
random_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

grouped_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

def run_cv(model_name, strategy):
    rows = []

    if strategy == "Random":
        iterator = random_cv.split(X_pixels, y)
    elif strategy == "Hostname_Grouped":
        iterator = grouped_cv.split(X_pixels, y, groups=groups)
    else:
        raise ValueError(strategy)

    for fold, (tr_idx, va_idx) in enumerate(iterator, start=1):

        model = make_model(model_name)
        model.fit(X_pixels[tr_idx], y[tr_idx])

        prob = model.predict_proba(X_pixels[va_idx])[:, 1]

        metrics = compute_metrics(y[va_idx], prob)

        train_hosts = set(groups[tr_idx])
        val_hosts = set(groups[va_idx])
        overlap = len(train_hosts & val_hosts)

        metrics.update({
            "model": model_name,
            "strategy": strategy,
            "fold": fold,
            "hostname_overlap": overlap,
            "train_n": len(tr_idx),
            "val_n": len(va_idx)
        })

        rows.append(metrics)

        print(
            f"{model_name:9s} | {strategy:16s} | Fold {fold} | "
            f"AUC={metrics['auc']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"MCC={metrics['mcc']:.4f} | "
            f"Host overlap={overlap}"
        )

    return pd.DataFrame(rows)

cv_frames = []

for model_name in ["XGBoost", "LightGBM"]:
    cv_frames.append(run_cv(model_name, "Random"))
    cv_frames.append(run_cv(model_name, "Hostname_Grouped"))

cv_results = pd.concat(cv_frames, ignore_index=True)

display(cv_results.round(4))

In [ ]:
cv_summary = (
    cv_results
    .groupby(["model", "strategy"])
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        mean_auc=("auc", "mean"),
        std_auc=("auc", "std"),
        mean_mcc=("mcc", "mean"),
        std_mcc=("mcc", "std"),
        mean_hostname_overlap=("hostname_overlap", "mean")
    )
    .reset_index()
)

display(cv_summary.round(4))

inflation_rows = []

for model_name in ["XGBoost", "LightGBM"]:
    rand = cv_summary[
        (cv_summary["model"] == model_name) &
        (cv_summary["strategy"] == "Random")
    ].iloc[0]

    grp = cv_summary[
        (cv_summary["model"] == model_name) &
        (cv_summary["strategy"] == "Hostname_Grouped")
    ].iloc[0]

    inflation_rows.append({
        "model": model_name,
        "F1_random_minus_grouped": rand["mean_f1"] - grp["mean_f1"],
        "AUC_random_minus_grouped": rand["mean_auc"] - grp["mean_auc"],
        "MCC_random_minus_grouped": rand["mean_mcc"] - grp["mean_mcc"],
        "random_mean_hostname_overlap": rand["mean_hostname_overlap"],
        "grouped_mean_hostname_overlap": grp["mean_hostname_overlap"]
    })

inflation_df = pd.DataFrame(inflation_rows)

print("\\nRandom-split inflation:")
display(inflation_df.round(4))

In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

dev_idx, test_idx = next(
    gss.split(X_pixels, y, groups=groups)
)

dev_hosts = set(groups[dev_idx])
test_hosts = set(groups[test_idx])

print("Development samples:", len(dev_idx))
print("Test samples:", len(test_idx))
print("Hostname overlap:", len(dev_hosts & test_hosts))

assert len(dev_hosts & test_hosts) == 0

In [ ]:
holdout_rows = []

for model_name in ["XGBoost", "LightGBM"]:

    model = make_model(model_name)
    model.fit(X_pixels[dev_idx], y[dev_idx])

    prob = model.predict_proba(X_pixels[test_idx])[:,1]
    metrics = compute_metrics(y[test_idx], prob)

    metrics["model"] = model_name
    metrics["dev_n"] = len(dev_idx)
    metrics["test_n"] = len(test_idx)
    metrics["hostname_overlap"] = len(dev_hosts & test_hosts)

    holdout_rows.append(metrics)

holdout_results = pd.DataFrame(holdout_rows)

display(
    holdout_results[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auc",
            "mcc",
            "tn","fp","fn","tp",
            "hostname_overlap"
        ]
    ].round(4)
)

In [ ]:
published = pd.DataFrame([
    {
        "model": "XGBoost",
        "published_accuracy": 0.8258,
        "published_precision": 0.8332,
        "published_recall": 0.8041,
        "published_f1": 0.8184,
        "published_auc": 0.9083
    },
    {
        "model": "LightGBM",
        "published_accuracy": 0.8293,
        "published_precision": 0.8394,
        "published_recall": 0.8041,
        "published_f1": 0.8214,
        "published_auc": 0.9106
    }
])

holdout_compare = published.merge(
    holdout_results[
        ["model","accuracy","precision","recall","f1","auc","mcc"]
    ],
    on="model",
    how="left"
)

holdout_compare["delta_auc_grouped_vs_published"] = (
    holdout_compare["auc"] - holdout_compare["published_auc"]
)

holdout_compare["delta_f1_grouped_vs_published"] = (
    holdout_compare["f1"] - holdout_compare["published_f1"]
)

display(holdout_compare.round(4))

In [ ]:
plot_df = cv_summary.copy()

x = np.arange(len(["XGBoost","LightGBM"]))
width = 0.35

for metric, ylabel in [("mean_auc","AUC"), ("mean_f1","F1"), ("mean_mcc","MCC")]:

    random_vals = []
    grouped_vals = []

    for model_name in ["XGBoost","LightGBM"]:
        random_vals.append(
            plot_df[
                (plot_df["model"] == model_name) &
                (plot_df["strategy"] == "Random")
            ][metric].iloc[0]
        )

        grouped_vals.append(
            plot_df[
                (plot_df["model"] == model_name) &
                (plot_df["strategy"] == "Hostname_Grouped")
            ][metric].iloc[0]
        )

    fig, ax = plt.subplots(figsize=(6,4))
    ax.bar(x - width/2, random_vals, width, label="Random CV")
    ax.bar(x + width/2, grouped_vals, width, label="Hostname-grouped CV")
    ax.set_xticks(x)
    ax.set_xticklabels(["XGBoost","LightGBM"])
    ax.set_ylabel(ylabel)
    ax.set_title(f"Raw-pixel baseline: {ylabel}")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
OUT = Path("experiment1_rawpixel_hostname_disjoint_results")
OUT.mkdir(exist_ok=True)

clean_df[
    ["original_index","url","normalized_url","hostname","label"]
].to_csv(
    OUT / "mapped_clean_samples.csv",
    index=False
)

cv_results.to_csv(
    OUT / "rawpixel_fold_results.csv",
    index=False
)

cv_summary.to_csv(
    OUT / "rawpixel_cv_summary.csv",
    index=False
)

inflation_df.to_csv(
    OUT / "rawpixel_random_split_inflation.csv",
    index=False
)

holdout_results.to_csv(
    OUT / "rawpixel_grouped_holdout_results.csv",
    index=False
)

holdout_compare.to_csv(
    OUT / "rawpixel_published_vs_grouped.csv",
    index=False
)

split_assignment = clean_df[
    ["original_index","hostname","label"]
].copy()

split_assignment["split"] = "development"
split_assignment.iloc[test_idx, split_assignment.columns.get_loc("split")] = "test"

split_assignment.to_csv(
    OUT / "hostname_grouped_split_assignment.csv",
    index=False
)

print("Saved to:", OUT.resolve())

for f in OUT.iterdir():
    print("-", f.name)

Conventional random splitting inflates the apparent generalization performance of raw-pixel quishing detectors, while hostname-disjoint evaluation reveals a measurable performance degradation.

# Experiment 2 — Lexical–Structural Signal Attribution and Incremental Value
## Research purpose
This experiment addresses the central question of the paper:
 What are QR structural detectors actually learning, and do QR structural features provide predictive information beyond the encoded URL?
## Part A — Signal attribution
We quantify how strongly QR structural features are related to lexical/payload properties using:
1. Cross-block Spearman correlations
2. Grouped out-of-fold prediction of each structural feature from lexical features
If lexical features can predict structural features well, this supports the interpretation that part of the structural signal reflects encoded payload properties.
## Part B — Incremental predictive value
Under the same hostname-disjoint evaluation protocol, we compare:
- Lexical
- Structural
- Structural intrinsic (excluding QR version)
- Lexical + Structural
- Lexical + Structural intrinsic
The same XGBoost configuration is used across representations so that the comparison isolates representation value instead of hyperparameter differences.
### We also perform:
- paired fold comparisons
- Wilcoxon signed-rank tests
- hostname-grouped bootstrap confidence intervals
- an exploratory residualized-structure analysis
### Important principle
This experiment does not try to prove that structural information is useless.
It asks a narrower question:
How much predictive information remains in QR structure after lexical/payload information is already available?

In [ ]:
%pip install numpy pandas scipy scikit-learn xgboost matplotlib qrcode[pil] opencv-python tldextract

In [ ]:
import os
import re
import gc
import pickle
import warnings
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse, urlunparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, wilcoxon

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    r2_score,
    mean_absolute_error
)
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBClassifier

import qrcode
from qrcode.constants import ERROR_CORRECT_M

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
N_BOOT = 1000

np.random.seed(SEED)

DATA_DIR = Path(r"C:\Users\LENOVO\FYP\QuishingDataset")

print("Ready.")

In [ ]:
FULL_FEATURE_CSV = Path("qr_full_feature_dataset_v2.csv")
EXP1_MAPPING = Path(
    "experiment1_rawpixel_hostname_disjoint_results"
) / "mapped_clean_samples.csv"

if FULL_FEATURE_CSV.exists():

    print("Loading existing full feature dataset:")
    print(FULL_FEATURE_CSV.resolve())

    df_existing = pd.read_csv(FULL_FEATURE_CSV)

    base_cols = [
        "original_index",
        "url",
        "normalized_url",
        "hostname",
        "label"
    ]

    missing_base = [
        c for c in base_cols
        if c not in df_existing.columns
    ]

    if missing_base:
        raise ValueError(
            f"Existing CSV is missing columns: {missing_base}"
        )

    base_df = df_existing[base_cols].copy()
    USE_EXISTING_FEATURES = True

elif EXP1_MAPPING.exists():

    print("Loading Experiment 1 mapping:")
    print(EXP1_MAPPING.resolve())

    base_df = pd.read_csv(EXP1_MAPPING)
    USE_EXISTING_FEATURES = False

else:

    print("No cached clean dataset found.")
    print("Reconstructing from official QR pickles...")

    import cv2

    QR_FILE = DATA_DIR / "qr_codes_29.pickle"
    LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

    with open(QR_FILE, "rb") as f:
        qr_codes = np.asarray(pickle.load(f))

    with open(LABEL_FILE, "rb") as f:
        labels = np.asarray(pickle.load(f)).astype(int)

    detector = cv2.QRCodeDetector()
    decoded_urls = []

    for i, qr in enumerate(qr_codes):

        img = (
            (qr * 255).astype(np.uint8)
            if qr.max() <= 1
            else qr.astype(np.uint8)
        )

        img = cv2.copyMakeBorder(
            img, 4, 4, 4, 4,
            cv2.BORDER_CONSTANT,
            value=255
        )

        img = cv2.resize(
            img,
            None,
            fx=10,
            fy=10,
            interpolation=cv2.INTER_NEAREST
        )

        data, _, _ = detector.detectAndDecode(img)
        decoded_urls.append(data if data else None)

        if (i + 1) % 1000 == 0:
            print(f"Decoded {i+1}/{len(qr_codes)}")

    tmp = pd.DataFrame({
        "original_index": np.arange(len(qr_codes)),
        "url": decoded_urls,
        "label": labels
    })

    def normalize_url(url):
        url = str(url).strip()

        parse_url = url
        if not url.lower().startswith(("http://", "https://")):
            parse_url = "http://" + url

        parsed = urlparse(parse_url)
        hostname = parsed.hostname.lower() if parsed.hostname else ""

        if hostname.startswith("www."):
            hostname = hostname[4:]

        try:
            port = parsed.port
        except ValueError:
            port = None

        netloc = hostname
        if port is not None:
            netloc += f":{port}"

        return urlunparse((
            parsed.scheme.lower(),
            netloc,
            parsed.path,
            "",
            parsed.query,
            ""
        ))

    def extract_hostname(url):
        parse_url = str(url).strip()

        if not parse_url.lower().startswith(("http://", "https://")):
            parse_url = "http://" + parse_url

        h = urlparse(parse_url).hostname

        if h is None:
            return None

        h = h.lower()

        if h.startswith("www."):
            h = h[4:]

        return h

    base_df = tmp.dropna(subset=["url"]).copy()
    base_df["normalized_url"] = base_df["url"].apply(normalize_url)

    base_df = (
        base_df
        .drop_duplicates(subset=["normalized_url"])
        .copy()
    )

    base_df["hostname"] = base_df["url"].apply(extract_hostname)
    base_df = base_df.dropna(subset=["hostname"]).reset_index(drop=True)

    USE_EXISTING_FEATURES = False

print("Samples:", len(base_df))
print("Class counts:")
print(base_df["label"].value_counts())
print("Unique hostnames:", base_df["hostname"].nunique())

In [ ]:
try:
    import tldextract

    # Avoid unexpected online Public Suffix List download.
    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

except Exception:
    domain_extractor = None


SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def extract_lexical_features(url):

    raw_url = str(url).strip()
    lower_url = raw_url.lower()

    scheme_present = int(
        lower_url.startswith(("http://", "https://"))
    )

    parse_url = raw_url

    if not scheme_present:
        parse_url = "http://" + raw_url

    parsed = urlparse(parse_url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""
    fragment = parsed.fragment or ""

    url_length = len(raw_url)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    if domain_extractor is not None:

        ext = domain_extractor(hostname)

        if ext.subdomain:
            num_subdomains = len(
                [
                    x
                    for x in ext.subdomain.split(".")
                    if x
                ]
            )
        else:
            num_subdomains = 0

    else:
        # Fallback only if tldextract is unavailable.
        parts = [x for x in hostname.split(".") if x]
        num_subdomains = max(len(parts) - 2, 0)

    suspicious_keyword_count = sum(
        keyword in lower_url
        for keyword in SUSPICIOUS_KEYWORDS
    )

    try:
        contains_port = int(
            parsed.port is not None
        )
    except ValueError:
        contains_port = 0

    return {
        "url_length": url_length,
        "hostname_length": len(hostname),
        "path_length": len(path),
        "query_length": len(query),

        "num_dots": raw_url.count("."),
        "num_hyphens": raw_url.count("-"),
        "num_digits": num_digits,
        "num_special_chars": num_special_chars,

        "digit_ratio":
            num_digits / max(url_length, 1),

        "special_char_ratio":
            num_special_chars / max(url_length, 1),

        "num_subdomains":
            num_subdomains,

        "has_query":
            int(bool(query)),

        "has_fragment":
            int(bool(fragment)),

        "contains_at":
            int("@" in raw_url),

        "contains_port":
            contains_port,

        "suspicious_keyword_count":
            suspicious_keyword_count
    }

In [ ]:
def generate_standardized_qr(url):

    qr = qrcode.QRCode(
        version=None,
        error_correction=ERROR_CORRECT_M,
        box_size=1,
        border=4
    )

    qr.add_data(url)
    qr.make(fit=True)

    matrix = np.array(
        qr.get_matrix(),
        dtype=np.uint8
    )

    return matrix, qr.version


def remove_border(matrix, border=4):

    if border == 0:
        return matrix

    return matrix[
        border:-border,
        border:-border
    ]


def transition_features(matrix):

    m = remove_border(matrix)

    horizontal = (
        m[:, 1:] != m[:, :-1]
    )

    vertical = (
        m[1:, :] != m[:-1, :]
    )

    h_density = horizontal.mean()
    v_density = vertical.mean()

    total_transitions = (
        horizontal.sum()
        +
        vertical.sum()
    )

    total_pairs = (
        horizontal.size
        +
        vertical.size
    )

    return (
        h_density,
        v_density,
        total_transitions / total_pairs
    )


def get_runs(arr):

    runs = []

    if len(arr) == 0:
        return runs

    current = arr[0]
    length = 1

    for value in arr[1:]:

        if value == current:
            length += 1

        else:
            runs.append(length)
            current = value
            length = 1

    runs.append(length)

    return runs


def run_length_features(matrix):

    m = remove_border(matrix)

    horizontal_runs = []
    vertical_runs = []

    for row in m:
        horizontal_runs.extend(
            get_runs(row)
        )

    for col in m.T:
        vertical_runs.extend(
            get_runs(col)
        )

    return {
        "mean_horizontal_run":
            np.mean(horizontal_runs),

        "std_horizontal_run":
            np.std(horizontal_runs),

        "max_horizontal_run":
            np.max(horizontal_runs),

        "mean_vertical_run":
            np.mean(vertical_runs),

        "std_vertical_run":
            np.std(vertical_runs),

        "max_vertical_run":
            np.max(vertical_runs)
    }


def pattern_2x2_features(matrix):

    m = remove_border(matrix)
    patterns = []

    for i in range(m.shape[0] - 1):

        for j in range(m.shape[1] - 1):

            block = m[
                i:i+2,
                j:j+2
            ]

            patterns.append(
                tuple(block.flatten())
            )

    counts = Counter(patterns)
    total = len(patterns)

    probs = (
        np.array(
            list(counts.values()),
            dtype=float
        )
        /
        total
    )

    entropy = -np.sum(
        probs * np.log2(probs)
    )

    diversity = len(counts) / 16

    return entropy, diversity


def extract_structural_features(
    matrix,
    qr_version
):

    m = remove_border(matrix)

    h_t, v_t, overall_t = (
        transition_features(matrix)
    )

    run_features = (
        run_length_features(matrix)
    )

    entropy_2x2, diversity_2x2 = (
        pattern_2x2_features(matrix)
    )

    features = {
        "qr_version":
            qr_version,

        "qr_matrix_size":
            m.shape[0],

        "black_ratio":
            np.mean(m == 1),

        "horizontal_transition_density":
            h_t,

        "vertical_transition_density":
            v_t,

        "overall_transition_density":
            overall_t,

        "pattern_2x2_entropy":
            entropy_2x2,

        "pattern_2x2_diversity":
            diversity_2x2
    }

    features.update(run_features)

    return features

In [ ]:
if USE_EXISTING_FEATURES:

    print(
        "Using lexical and structural values "
        "already generated by your original notebook."
    )

    df_all = df_existing.copy()

else:

    print("Generating lexical features...")

    lexical_df = pd.DataFrame(
        base_df["url"]
        .apply(extract_lexical_features)
        .tolist()
    )

    print("Generating standardized QR structural features...")

    structural_records = []

    for i, url in enumerate(base_df["url"]):

        matrix, version = (
            generate_standardized_qr(url)
        )

        structural_records.append(
            extract_structural_features(
                matrix,
                version
            )
        )

        if (i + 1) % 1000 == 0:
            print(
                f"Generated {i+1}/{len(base_df)}"
            )

    structural_df = pd.DataFrame(
        structural_records
    )

    # Match original notebook
    structural_df = structural_df.drop(
        columns=["pattern_2x2_diversity"]
    )

    df_all = pd.concat(
        [
            base_df.reset_index(drop=True),
            lexical_df.reset_index(drop=True),
            structural_df.reset_index(drop=True)
        ],
        axis=1
    )

    df_all.to_csv(
        "experiment2_full_feature_dataset.csv",
        index=False
    )

print("Final shape:", df_all.shape)

In [ ]:
LEXICAL_ALL = [
    "url_length",
    "hostname_length",
    "path_length",
    "query_length",
    "num_dots",
    "num_hyphens",
    "num_digits",
    "num_special_chars",
    "digit_ratio",
    "special_char_ratio",
    "num_subdomains",
    "has_query",
    "has_fragment",
    "contains_at",
    "contains_port",
    "suspicious_keyword_count"
]

STRUCTURAL_ALL = [
    "qr_version",
    "qr_matrix_size",
    "black_ratio",
    "horizontal_transition_density",
    "vertical_transition_density",
    "overall_transition_density",
    "pattern_2x2_entropy",
    "mean_horizontal_run",
    "std_horizontal_run",
    "max_horizontal_run",
    "mean_vertical_run",
    "std_vertical_run",
    "max_vertical_run"
]

# Original redundancy-filtered variables from your notebook.
REDUNDANT_FEATURES = [
    "qr_matrix_size",
    "has_query",
    "mean_horizontal_run",
    "mean_vertical_run",
    "digit_ratio"
]

lexical_reduced = [
    f
    for f in LEXICAL_ALL
    if (
        f in df_all.columns
        and
        f not in REDUNDANT_FEATURES
    )
]

structural_reduced = [
    f
    for f in STRUCTURAL_ALL
    if (
        f in df_all.columns
        and
        f not in REDUNDANT_FEATURES
    )
]

structural_intrinsic = [
    f
    for f in structural_reduced
    if f != "qr_version"
]

lexical_plus_structural = (
    lexical_reduced
    +
    structural_reduced
)

lexical_plus_intrinsic = (
    lexical_reduced
    +
    structural_intrinsic
)

print("Lexical reduced:", len(lexical_reduced))
print(lexical_reduced)

print("\nStructural reduced:", len(structural_reduced))
print(structural_reduced)

print("\nStructural intrinsic:", len(structural_intrinsic))
print(structural_intrinsic)

print(
    "\nCombined lexical + structural:",
    len(lexical_plus_structural)
)

In [ ]:
required_cols = list(dict.fromkeys(
    lexical_plus_structural
    +
    ["hostname", "label"]
))

missing_cols = [
    c
    for c in required_cols
    if c not in df_all.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

print(
    "Missing feature values:",
    df_all[
        lexical_plus_structural
    ].isna().sum().sum()
)

print(
    "Infinite feature values:",
    np.isinf(
        df_all[
            lexical_plus_structural
        ].astype(float)
    ).sum().sum()
)

print(
    "Class counts:"
)
print(
    df_all["label"].value_counts()
)

print(
    "Unique hostnames:",
    df_all["hostname"].nunique()
)

In [ ]:
# part A -- Signal Attribution
cross_corr_records = []

for lex in lexical_reduced:

    for structural in structural_reduced:

        rho, p = spearmanr(
            df_all[lex],
            df_all[structural],
            nan_policy="omit"
        )

        cross_corr_records.append({
            "lexical_feature": lex,
            "structural_feature": structural,
            "spearman_rho": rho,
            "abs_rho": abs(rho),
            "p_value": p
        })

cross_corr_df = (
    pd.DataFrame(cross_corr_records)
    .sort_values(
        "abs_rho",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top lexical–structural associations:")
display(
    cross_corr_df.head(25).round(4)
)

In [ ]:
cross_matrix = pd.DataFrame(
    index=lexical_reduced,
    columns=structural_reduced,
    dtype=float
)

for lex in lexical_reduced:
    for structural in structural_reduced:
        cross_matrix.loc[lex, structural] = (
            spearmanr(
                df_all[lex],
                df_all[structural],
                nan_policy="omit"
            )[0]
        )

fig, ax = plt.subplots(
    figsize=(12, 8)
)

im = ax.imshow(
    cross_matrix.values.astype(float),
    aspect="auto",
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    np.arange(len(structural_reduced))
)

ax.set_xticklabels(
    structural_reduced,
    rotation=70,
    ha="right"
)

ax.set_yticks(
    np.arange(len(lexical_reduced))
)

ax.set_yticklabels(
    lexical_reduced
)

ax.set_title(
    "Lexical–Structural Spearman Correlations"
)

fig.colorbar(
    im,
    ax=ax,
    label="Spearman rho"
)

plt.tight_layout()
plt.show()

In [ ]:
groups_all = (
    df_all["hostname"]
    .astype(str)
    .reset_index(drop=True)
)

y_label_all = (
    df_all["label"]
    .astype(int)
    .reset_index(drop=True)
)

X_lex_all = (
    df_all[
        lexical_reduced
    ]
    .reset_index(drop=True)
)

structural_predictability_rows = []

attr_cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

# Precompute fold assignments once so every target
# uses exactly the same grouped folds.
attr_splits = list(
    attr_cv.split(
        X_lex_all,
        y_label_all,
        groups=groups_all
    )
)

for target in structural_reduced:

    y_target = (
        df_all[target]
        .astype(float)
        .reset_index(drop=True)
    )

    oof_pred = np.full(
        len(df_all),
        np.nan
    )

    for fold, (tr_idx, va_idx) in enumerate(
        attr_splits,
        start=1
    ):

        reg = RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=3,
            random_state=SEED + fold,
            n_jobs=-1
        )

        reg.fit(
            X_lex_all.iloc[tr_idx],
            y_target.iloc[tr_idx]
        )

        oof_pred[va_idx] = reg.predict(
            X_lex_all.iloc[va_idx]
        )

    target_range = (
        y_target.max()
        -
        y_target.min()
    )

    mae = mean_absolute_error(
        y_target,
        oof_pred
    )

    nmae = (
        mae / target_range
        if target_range > 0
        else np.nan
    )

    structural_predictability_rows.append({
        "structural_feature": target,
        "oof_r2_from_lexical":
            r2_score(
                y_target,
                oof_pred
            ),
        "oof_mae":
            mae,
        "normalized_mae":
            nmae
    })

structural_predictability_df = (
    pd.DataFrame(
        structural_predictability_rows
    )
    .sort_values(
        "oof_r2_from_lexical",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    structural_predictability_df.round(4)
)

In [ ]:
predictability_summary = pd.DataFrame([{
    "mean_structural_R2_from_lexical":
        structural_predictability_df[
            "oof_r2_from_lexical"
        ].mean(),

    "median_structural_R2_from_lexical":
        structural_predictability_df[
            "oof_r2_from_lexical"
        ].median(),

    "max_structural_R2_from_lexical":
        structural_predictability_df[
            "oof_r2_from_lexical"
        ].max(),

    "qr_version_R2_from_lexical":
        structural_predictability_df.loc[
            structural_predictability_df[
                "structural_feature"
            ] == "qr_version",
            "oof_r2_from_lexical"
        ].iloc[0]
        if "qr_version" in structural_reduced
        else np.nan
}])

display(
    predictability_summary.round(4)
)

In [ ]:
# part B -- incremental predictive value
feature_universe = list(
    dict.fromkeys(
        lexical_plus_structural
    )
)

X = (
    df_all[
        feature_universe
    ]
    .copy()
)

y = (
    df_all["label"]
    .astype(int)
    .copy()
)

groups = (
    df_all["hostname"]
    .astype(str)
    .copy()
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

dev_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_dev = (
    X.iloc[dev_idx]
    .reset_index(drop=True)
)

X_test = (
    X.iloc[test_idx]
    .reset_index(drop=True)
)

y_dev = (
    y.iloc[dev_idx]
    .reset_index(drop=True)
)

y_test = (
    y.iloc[test_idx]
    .reset_index(drop=True)
)

groups_dev = (
    groups.iloc[dev_idx]
    .reset_index(drop=True)
)

groups_test = (
    groups.iloc[test_idx]
    .reset_index(drop=True)
)

print("Development samples:", len(X_dev))
print("Test samples:", len(X_test))

host_overlap = (
    set(groups_dev)
    &
    set(groups_test)
)

print(
    "Hostname overlap:",
    len(host_overlap)
)

assert len(host_overlap) == 0

In [ ]:
FIXED_XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 1.0
}


def make_xgb(seed=SEED):

    return XGBClassifier(
        **FIXED_XGB_PARAMS,
        eval_metric="logloss",
        random_state=seed,
        n_jobs=-1
    )


feature_sets = {
    "Lexical":
        lexical_reduced,

    "Structural":
        structural_reduced,

    "Structural_intrinsic":
        structural_intrinsic,

    "Lexical_plus_Structural":
        lexical_plus_structural,

    "Lexical_plus_Intrinsic":
        lexical_plus_intrinsic
}

for k, v in feature_sets.items():
    print(
        f"{k}: {len(v)} features"
    )

In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_splits = list(
    sgkf.split(
        X_dev,
        y_dev,
        groups=groups_dev
    )
)


def metric_dict(
    y_true,
    prob
):

    pred = (
        prob >= 0.5
    ).astype(int)

    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                pred
            )
    }


fold_records = []
oof_prediction_records = []

for set_name, features in (
    feature_sets.items()
):

    print(
        "\n"
        +
        "=" * 72
    )

    print(
        set_name
    )

    print(
        "=" * 72
    )

    oof_prob = np.full(
        len(X_dev),
        np.nan
    )

    for fold, (
        tr_idx,
        va_idx
    ) in enumerate(
        fold_splits,
        start=1
    ):

        model = make_xgb(
            SEED + fold
        )

        model.fit(
            X_dev[
                features
            ].iloc[tr_idx],
            y_dev.iloc[tr_idx]
        )

        prob = (
            model.predict_proba(
                X_dev[
                    features
                ].iloc[va_idx]
            )[:, 1]
        )

        oof_prob[va_idx] = prob

        metrics = metric_dict(
            y_dev.iloc[va_idx],
            prob
        )

        train_hosts = set(
            groups_dev.iloc[tr_idx]
        )

        val_hosts = set(
            groups_dev.iloc[va_idx]
        )

        metrics.update({
            "feature_set":
                set_name,

            "fold":
                fold,

            "hostname_overlap":
                len(
                    train_hosts
                    &
                    val_hosts
                )
        })

        fold_records.append(
            metrics
        )

        print(
            f"Fold {fold} | "
            f"F1={metrics['f1']:.4f} | "
            f"AUC={metrics['auc']:.4f} | "
            f"MCC={metrics['mcc']:.4f} | "
            f"overlap={metrics['hostname_overlap']}"
        )

    assert not np.isnan(
        oof_prob
    ).any()

    for i in range(
        len(X_dev)
    ):

        oof_prediction_records.append({
            "feature_set":
                set_name,

            "dev_row":
                i,

            "hostname":
                groups_dev.iloc[i],

            "label":
                int(
                    y_dev.iloc[i]
                ),

            "probability":
                float(
                    oof_prob[i]
                )
        })


fold_results = pd.DataFrame(
    fold_records
)

oof_predictions = pd.DataFrame(
    oof_prediction_records
)

display(
    fold_results.round(4)
)

In [ ]:
cv_summary = (
    fold_results
    .groupby(
        "feature_set"
    )
    .agg(
        mean_accuracy=(
            "accuracy",
            "mean"
        ),
        std_accuracy=(
            "accuracy",
            "std"
        ),
        mean_precision=(
            "precision",
            "mean"
        ),
        mean_recall=(
            "recall",
            "mean"
        ),
        mean_f1=(
            "f1",
            "mean"
        ),
        std_f1=(
            "f1",
            "std"
        ),
        mean_auc=(
            "auc",
            "mean"
        ),
        std_auc=(
            "auc",
            "std"
        ),
        mean_mcc=(
            "mcc",
            "mean"
        ),
        std_mcc=(
            "mcc",
            "std"
        ),
        mean_hostname_overlap=(
            "hostname_overlap",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "mean_auc",
        ascending=False
    )
)

display(
    cv_summary.round(4)
)

In [ ]:
comparisons = [
    (
        "Lexical_plus_Structural",
        "Lexical"
    ),
    (
        "Lexical_plus_Intrinsic",
        "Lexical"
    )
]

delta_rows = []

for augmented, base in comparisons:

    aug = (
        fold_results[
            fold_results[
                "feature_set"
            ] == augmented
        ]
        .sort_values("fold")
        .reset_index(drop=True)
    )

    ref = (
        fold_results[
            fold_results[
                "feature_set"
            ] == base
        ]
        .sort_values("fold")
        .reset_index(drop=True)
    )

    for metric in [
        "f1",
        "auc",
        "mcc"
    ]:

        diffs = (
            aug[metric]
            -
            ref[metric]
        )

        delta_rows.append({
            "comparison":
                f"{augmented} - {base}",

            "metric":
                metric,

            "mean_delta":
                diffs.mean(),

            "std_delta":
                diffs.std(ddof=1),

            "wins_augmented":
                int(
                    (diffs > 0).sum()
                ),

            "losses_augmented":
                int(
                    (diffs < 0).sum()
                )
        })

delta_df = pd.DataFrame(
    delta_rows
)

display(
    delta_df.round(4)
)

In [ ]:
wilcoxon_rows = []

for augmented, base in comparisons:

    aug = (
        fold_results[
            fold_results[
                "feature_set"
            ] == augmented
        ]
        .sort_values("fold")
    )

    ref = (
        fold_results[
            fold_results[
                "feature_set"
            ] == base
        ]
        .sort_values("fold")
    )

    for metric in [
        "f1",
        "auc",
        "mcc"
    ]:

        x = (
            aug[metric]
            .to_numpy()
        )

        y0 = (
            ref[metric]
            .to_numpy()
        )

        try:

            stat, p = wilcoxon(
                x,
                y0,
                alternative="two-sided"
            )

        except ValueError:

            stat, p = (
                np.nan,
                1.0
            )

        wilcoxon_rows.append({
            "comparison":
                f"{augmented} - {base}",

            "metric":
                metric,

            "wilcoxon_stat":
                stat,

            "p_value":
                p
        })

wilcoxon_df = pd.DataFrame(
    wilcoxon_rows
)

display(
    wilcoxon_df.round(4)
)

print(
    "\nNote: with only five folds, "
    "Wilcoxon p-values have limited resolution. "
    "The grouped bootstrap below is therefore "
    "more informative for uncertainty."
)

In [ ]:
def get_oof_table(
    feature_set_name
):

    temp = (
        oof_predictions[
            oof_predictions[
                "feature_set"
            ]
            ==
            feature_set_name
        ]
        [
            [
                "dev_row",
                "hostname",
                "label",
                "probability"
            ]
        ]
        .copy()
        .sort_values(
            "dev_row"
        )
        .reset_index(drop=True)
    )

    return temp


def grouped_bootstrap_delta(
    augmented_name,
    base_name,
    n_boot=N_BOOT,
    seed=SEED
):

    aug = get_oof_table(
        augmented_name
    )

    ref = get_oof_table(
        base_name
    )

    assert np.array_equal(
        aug["dev_row"].values,
        ref["dev_row"].values
    )

    assert np.array_equal(
        aug["label"].values,
        ref["label"].values
    )

    hosts = (
        aug["hostname"]
        .unique()
    )

    host_to_rows = {
        h:
        np.where(
            aug["hostname"].values
            == h
        )[0]
        for h in hosts
    }

    rng = np.random.default_rng(
        seed
    )

    rows = []

    for b in range(
        n_boot
    ):

        sampled_hosts = rng.choice(
            hosts,
            size=len(hosts),
            replace=True
        )

        sampled_indices = np.concatenate(
            [
                host_to_rows[h]
                for h in sampled_hosts
            ]
        )

        yb = (
            aug["label"]
            .values[
                sampled_indices
            ]
        )

        # Skip rare bootstrap draw with one class only.
        if len(
            np.unique(yb)
        ) < 2:
            continue

        aug_prob = (
            aug[
                "probability"
            ]
            .values[
                sampled_indices
            ]
        )

        ref_prob = (
            ref[
                "probability"
            ]
            .values[
                sampled_indices
            ]
        )

        aug_m = metric_dict(
            yb,
            aug_prob
        )

        ref_m = metric_dict(
            yb,
            ref_prob
        )

        rows.append({
            "delta_auc":
                aug_m["auc"]
                -
                ref_m["auc"],

            "delta_f1":
                aug_m["f1"]
                -
                ref_m["f1"],

            "delta_mcc":
                aug_m["mcc"]
                -
                ref_m["mcc"]
        })

    boot = pd.DataFrame(
        rows
    )

    summary_rows = []

    for metric in [
        "delta_auc",
        "delta_f1",
        "delta_mcc"
    ]:

        vals = boot[
            metric
        ].values

        summary_rows.append({
            "comparison":
                f"{augmented_name} - {base_name}",

            "metric":
                metric,

            "bootstrap_mean":
                np.mean(vals),

            "ci_2.5":
                np.percentile(
                    vals,
                    2.5
                ),

            "ci_97.5":
                np.percentile(
                    vals,
                    97.5
                ),

            "probability_delta_gt_0":
                np.mean(
                    vals > 0
                )
        })

    return (
        boot,
        pd.DataFrame(
            summary_rows
        )
    )


bootstrap_all = []
bootstrap_summaries = []

for augmented, base in comparisons:

    boot, summary = (
        grouped_bootstrap_delta(
            augmented,
            base
        )
    )

    boot[
        "comparison"
    ] = (
        f"{augmented} - {base}"
    )

    bootstrap_all.append(
        boot
    )

    bootstrap_summaries.append(
        summary
    )


bootstrap_df = pd.concat(
    bootstrap_all,
    ignore_index=True
)

bootstrap_summary = pd.concat(
    bootstrap_summaries,
    ignore_index=True
)

display(
    bootstrap_summary.round(4)
)

In [ ]:
holdout_rows = []

for set_name, features in (
    feature_sets.items()
):

    model = make_xgb(
        SEED
    )

    model.fit(
        X_dev[
            features
        ],
        y_dev
    )

    prob = (
        model.predict_proba(
            X_test[
                features
            ]
        )[:, 1]
    )

    metrics = metric_dict(
        y_test,
        prob
    )

    metrics.update({
        "feature_set":
            set_name,

        "hostname_overlap":
            len(
                set(groups_dev)
                &
                set(groups_test)
            )
    })

    holdout_rows.append(
        metrics
    )


holdout_df = (
    pd.DataFrame(
        holdout_rows
    )
    .sort_values(
        "auc",
        ascending=False
    )
)

display(
    holdout_df.round(4)
)

In [ ]:
holdout_delta_rows = []

holdout_indexed = (
    holdout_df
    .set_index(
        "feature_set"
    )
)

for augmented, base in comparisons:

    for metric in [
        "f1",
        "auc",
        "mcc"
    ]:

        holdout_delta_rows.append({
            "comparison":
                f"{augmented} - {base}",

            "metric":
                metric,

            "delta":
                holdout_indexed.loc[
                    augmented,
                    metric
                ]
                -
                holdout_indexed.loc[
                    base,
                    metric
                ]
        })

holdout_delta_df = pd.DataFrame(
    holdout_delta_rows
)

display(
    holdout_delta_df.round(4)
)

In [ ]:
# part C -- Exploratory Conditional Analysis
residual_records = []

for fold, (
    tr_idx,
    va_idx
) in enumerate(
    fold_splits,
    start=1
):

    Xlex_tr = (
        X_dev[
            lexical_reduced
        ]
        .iloc[tr_idx]
        .reset_index(drop=True)
    )

    Xlex_va = (
        X_dev[
            lexical_reduced
        ]
        .iloc[va_idx]
        .reset_index(drop=True)
    )

    S_tr = (
        X_dev[
            structural_intrinsic
        ]
        .iloc[tr_idx]
        .reset_index(drop=True)
    )

    S_va = (
        X_dev[
            structural_intrinsic
        ]
        .iloc[va_idx]
        .reset_index(drop=True)
    )

    reg = RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=SEED + fold,
        n_jobs=-1
    )

    reg.fit(
        Xlex_tr,
        S_tr
    )

    S_hat_tr = reg.predict(
        Xlex_tr
    )

    S_hat_va = reg.predict(
        Xlex_va
    )

    R_tr = (
        S_tr.to_numpy()
        -
        S_hat_tr
    )

    R_va = (
        S_va.to_numpy()
        -
        S_hat_va
    )

    residual_names = [
        f"resid_{c}"
        for c in structural_intrinsic
    ]

    residual_tr_df = pd.DataFrame(
        R_tr,
        columns=residual_names
    )

    residual_va_df = pd.DataFrame(
        R_va,
        columns=residual_names
    )

    lexical_tr_df = (
        Xlex_tr
        .reset_index(drop=True)
    )

    lexical_va_df = (
        Xlex_va
        .reset_index(drop=True)
    )

    aug_tr = pd.concat(
        [
            lexical_tr_df,
            residual_tr_df
        ],
        axis=1
    )

    aug_va = pd.concat(
        [
            lexical_va_df,
            residual_va_df
        ],
        axis=1
    )

    lexical_model = make_xgb(
        SEED + fold
    )

    residual_model = make_xgb(
        SEED + fold
    )

    lexical_model.fit(
        lexical_tr_df,
        y_dev.iloc[tr_idx]
    )

    residual_model.fit(
        aug_tr,
        y_dev.iloc[tr_idx]
    )

    p_lex = (
        lexical_model.predict_proba(
            lexical_va_df
        )[:, 1]
    )

    p_resid = (
        residual_model.predict_proba(
            aug_va
        )[:, 1]
    )

    lex_m = metric_dict(
        y_dev.iloc[va_idx],
        p_lex
    )

    resid_m = metric_dict(
        y_dev.iloc[va_idx],
        p_resid
    )

    residual_records.append({
        "fold":
            fold,

        "lexical_auc":
            lex_m["auc"],

        "residual_augmented_auc":
            resid_m["auc"],

        "delta_auc":
            resid_m["auc"]
            -
            lex_m["auc"],

        "lexical_f1":
            lex_m["f1"],

        "residual_augmented_f1":
            resid_m["f1"],

        "delta_f1":
            resid_m["f1"]
            -
            lex_m["f1"],

        "lexical_mcc":
            lex_m["mcc"],

        "residual_augmented_mcc":
            resid_m["mcc"],

        "delta_mcc":
            resid_m["mcc"]
            -
            lex_m["mcc"]
    })


residual_analysis_df = pd.DataFrame(
    residual_records
)

display(
    residual_analysis_df.round(4)
)

In [ ]:
residual_summary = pd.DataFrame([{
    "mean_delta_auc":
        residual_analysis_df[
            "delta_auc"
        ].mean(),

    "std_delta_auc":
        residual_analysis_df[
            "delta_auc"
        ].std(ddof=1),

    "mean_delta_f1":
        residual_analysis_df[
            "delta_f1"
        ].mean(),

    "std_delta_f1":
        residual_analysis_df[
            "delta_f1"
        ].std(ddof=1),

    "mean_delta_mcc":
        residual_analysis_df[
            "delta_mcc"
        ].mean(),

    "std_delta_mcc":
        residual_analysis_df[
            "delta_mcc"
        ].std(ddof=1),

    "folds_auc_improved":
        int(
            (
                residual_analysis_df[
                    "delta_auc"
                ] > 0
            ).sum()
        ),

    "folds_f1_improved":
        int(
            (
                residual_analysis_df[
                    "delta_f1"
                ] > 0
            ).sum()
        )
}])

display(
    residual_summary.round(4)
)

In [ ]:
main_summary = (
    cv_summary[
        [
            "feature_set",
            "mean_f1",
            "std_f1",
            "mean_auc",
            "std_auc",
            "mean_mcc",
            "std_mcc"
        ]
    ]
    .copy()
)

main_summary = (
    main_summary
    .merge(
        holdout_df[
            [
                "feature_set",
                "f1",
                "auc",
                "mcc"
            ]
        ]
        .rename(
            columns={
                "f1":
                    "holdout_f1",
                "auc":
                    "holdout_auc",
                "mcc":
                    "holdout_mcc"
            }
        ),
        on="feature_set",
        how="left"
    )
)

display(
    main_summary.round(4)
)

In [ ]:
OUT = Path(
    "experiment2_signal_attribution_incremental_value"
)

OUT.mkdir(
    exist_ok=True
)

cross_corr_df.to_csv(
    OUT / "cross_block_spearman.csv",
    index=False
)

cross_matrix.to_csv(
    OUT / "cross_block_spearman_matrix.csv"
)

structural_predictability_df.to_csv(
    OUT / "structural_predictability_from_lexical.csv",
    index=False
)

predictability_summary.to_csv(
    OUT / "structural_predictability_summary.csv",
    index=False
)

fold_results.to_csv(
    OUT / "representation_grouped_cv_fold_results.csv",
    index=False
)

cv_summary.to_csv(
    OUT / "representation_grouped_cv_summary.csv",
    index=False
)

oof_predictions.to_csv(
    OUT / "representation_oof_predictions.csv",
    index=False
)

delta_df.to_csv(
    OUT / "incremental_fold_deltas.csv",
    index=False
)

wilcoxon_df.to_csv(
    OUT / "paired_wilcoxon_results.csv",
    index=False
)

bootstrap_df.to_csv(
    OUT / "hostname_grouped_bootstrap_samples.csv",
    index=False
)

bootstrap_summary.to_csv(
    OUT / "hostname_grouped_bootstrap_summary.csv",
    index=False
)

holdout_df.to_csv(
    OUT / "representation_holdout_results.csv",
    index=False
)

holdout_delta_df.to_csv(
    OUT / "holdout_incremental_deltas.csv",
    index=False
)

residual_analysis_df.to_csv(
    OUT / "residualized_structure_fold_results.csv",
    index=False
)

residual_summary.to_csv(
    OUT / "residualized_structure_summary.csv",
    index=False
)

main_summary.to_csv(
    OUT / "paper_ready_representation_summary.csv",
    index=False
)

print(
    "Saved to:",
    OUT.resolve()
)

for p in OUT.iterdir():
    print(
        "-",
        p.name
    )

The results suggest substantial information overlap between lexical and QR-structural representations. Structural features remain predictive in isolation, but do not provide incremental benefit once lexical URL information is available.

# Experiment 3 — External Generalization and Domain-Shift Verification
This notebook evaluates the frozen 10-feature lexical detector on an independent URL dataset. The external dataset is used only as an out-of-domain stress test: no retraining, feature reselection, or threshold tuning.
It audits dataset semantics, URL/hostname overlap, frozen-model generalization, and internal–external feature shift.
## Important: 
do not call this external quishing validation unless the external dataset source explicitly confirms that label 1 means phishing.

In [ ]:
%pip install numpy pandas scipy scikit-learn xgboost joblib matplotlib

In [ ]:
import os, json, joblib, warnings
from pathlib import Path
from urllib.parse import urlsplit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score, matthews_corrcoef, confusion_matrix
warnings.filterwarnings("ignore")
SEED=42
rng=np.random.default_rng(SEED)
EXTERNAL_CSV=Path(r"C:\Users\LENOVO\Desktop\FYP\archive\balanced_urls.csv")
MODEL_PATH=Path("FINAL_10feature_lightweight_xgb.pkl")
CONFIG_PATH=Path("FINAL_10feature_deployment_config.json")
INTERNAL_FEATURE_CSV_CANDIDATES=[Path("qr_full_feature_dataset_v2.csv"),Path("experiment2_full_feature_dataset.csv")]
OUT=Path("experiment3_external_generalization_domain_shift"); OUT.mkdir(exist_ok=True)
print(EXTERNAL_CSV, MODEL_PATH.resolve(), CONFIG_PATH.resolve(), sep="\n")

In [ ]:
EXTERNAL_DATASET_METADATA={
    "dataset_name":"balanced_urls.csv",
    "source_or_url":"UNVERIFIED",
    "positive_class_meaning":"UNVERIFIED",
    "negative_class_meaning":"UNVERIFIED",
    "collection_description":"UNVERIFIED",
    "publication_or_repository":"UNVERIFIED"
}
metadata_df=pd.DataFrame([EXTERNAL_DATASET_METADATA]); display(metadata_df)
semantics_verified=EXTERNAL_DATASET_METADATA["positive_class_meaning"]!="UNVERIFIED"
print("Semantics status:","VERIFIED" if semantics_verified else "UNVERIFIED")

In [ ]:
if not MODEL_PATH.exists(): raise FileNotFoundError(MODEL_PATH.resolve())
if not CONFIG_PATH.exists(): raise FileNotFoundError(CONFIG_PATH.resolve())
final_deployment_model=joblib.load(MODEL_PATH)
with open(CONFIG_PATH,"r",encoding="utf-8") as f: deployment_config=json.load(f)
final_deployment_features=list(deployment_config["features"])
print("Frozen features:", final_deployment_features)
print("Frozen threshold: 0.5")

In [ ]:
internal_csv=next((p for p in INTERNAL_FEATURE_CSV_CANDIDATES if p.exists()),None)
if internal_csv is None: raise FileNotFoundError("Need qr_full_feature_dataset_v2.csv or experiment2_full_feature_dataset.csv")
training_feature_data=pd.read_csv(internal_csv)
required=["url","hostname","label"]+final_deployment_features
missing=[c for c in required if c not in training_feature_data.columns]
if missing: raise ValueError(f"Missing internal columns: {missing}")
print("Internal:",internal_csv.resolve(),training_feature_data.shape)

In [ ]:
SUSPICIOUS_KEYWORDS=["login","verify","verification","account","secure","update","password","signin","confirm","payment","bank","billing","support"]
def ensure_scheme(url):
    url=str(url).strip(); return url if url.lower().startswith(("http://","https://")) else "http://"+url
def count_subdomains(hostname):
    hostname=str(hostname).strip().lower(); hostname=hostname[4:] if hostname.startswith("www.") else hostname
    parts=[p for p in hostname.split(".") if p]; return max(len(parts)-2,0)
def extract_deployment_features(url):
    raw_url=str(url).strip(); parsed=urlsplit(ensure_scheme(raw_url)); raw_hostname=(parsed.hostname or "").lower(); path=parsed.path or ""; query=parsed.query or ""
    url_length=len(raw_url); num_digits=sum(c.isdigit() for c in raw_url); num_dots=raw_url.count("."); num_hyphens=raw_url.count("-"); num_special_chars=sum(not c.isalnum() for c in raw_url)
    return {
        "num_subdomains":count_subdomains(raw_hostname),
        "suspicious_keyword_count":sum(k in raw_url.lower() for k in SUSPICIOUS_KEYWORDS),
        "num_digits":num_digits,"path_length":len(path),"hostname_length":len(raw_hostname),"url_length":url_length,"num_dots":num_dots,
        "special_char_ratio":num_special_chars/url_length if url_length>0 else 0.0,"num_hyphens":num_hyphens,"query_length":len(query)}
assert set(final_deployment_features)==set(extract_deployment_features("https://example.com/test?a=1").keys())
print("Extractor verified against frozen feature set.")

In [ ]:
mismatches=[]
for i,row in training_feature_data.iterrows():
    new=extract_deployment_features(row["url"])
    for f in final_deployment_features:
        old=row[f]; val=new[f]; ok=np.isclose(old,val,equal_nan=True) if isinstance(old,(float,np.floating)) else old==val
        if not ok: mismatches.append({"row":i,"feature":f,"stored_value":old,"recomputed_value":val})
deployment_audit_df=pd.DataFrame(mismatches)
print("Feature values checked:",len(training_feature_data)*len(final_deployment_features)); print("Mismatches:",len(deployment_audit_df))
if len(deployment_audit_df): display(deployment_audit_df.head(20))

In [ ]:
external_df=pd.read_csv(EXTERNAL_CSV)
required=["url","result"]
missing=[c for c in required if c not in external_df.columns]
if missing: raise ValueError(f"Missing external columns: {missing}")
audit_summary=pd.DataFrame([{"rows":len(external_df),"missing_urls":external_df.url.isna().sum(),"missing_labels":external_df.result.isna().sum(),"duplicate_urls":external_df.url.duplicated().sum(),"unique_urls":external_df.url.nunique(),"unique_labels":external_df.result.nunique()}]); display(audit_summary)
print(external_df.result.value_counts(dropna=False).sort_index()); print(external_df.result.value_counts(normalize=True).sort_index())
unexpected=set(external_df.result.dropna().unique())-{0,1}
if unexpected: raise ValueError(f"Unexpected labels: {unexpected}")

In [ ]:
def get_hostname(url):
    try:
        h=(urlsplit(ensure_scheme(str(url))).hostname or "").lower(); return h[4:] if h.startswith("www.") else h
    except: return ""
external_df["hostname"]=external_df.url.apply(get_hostname)
internal_hosts=training_feature_data.hostname.astype(str).str.lower().str.removeprefix("www.")
internal_urls=set(training_feature_data.url.astype(str)); original_hostnames=set(internal_hosts)
external_df["url_overlap"]=external_df.url.astype(str).isin(internal_urls)
external_df["hostname_overlap"]=external_df.hostname.isin(original_hostnames)
overlap_summary=pd.DataFrame([{"external_total":len(external_df),"exact_url_overlap_samples":int(external_df.url_overlap.sum()),"exact_url_overlap_rate":external_df.url_overlap.mean(),"hostname_overlap_samples":int(external_df.hostname_overlap.sum()),"hostname_overlap_rate":external_df.hostname_overlap.mean(),"unique_overlapping_hostnames":external_df.loc[external_df.hostname_overlap,"hostname"].nunique()}]); display(overlap_summary.round(6))

In [ ]:
external_unseen=external_df[(~external_df.hostname_overlap)&(~external_df.url_overlap)&(external_df.hostname!="")&external_df.url.notna()&external_df.result.notna()].copy().reset_index(drop=True)
unseen_summary=pd.DataFrame([{"external_total":len(external_df),"strict_unseen_host_samples":len(external_unseen),"strict_unseen_host_rate":len(external_unseen)/len(external_df),"class_0":int((external_unseen.result==0).sum()),"class_1":int((external_unseen.result==1).sum()),"unique_unseen_hosts":external_unseen.hostname.nunique()}]); display(unseen_summary.round(4))

In [ ]:
EXTERNAL_PER_CLASS=10000
parts=[]
for lab in [0,1]:
    sub=external_unseen[external_unseen.result==lab]; parts.append(sub.sample(n=min(EXTERNAL_PER_CLASS,len(sub)),random_state=SEED+lab))
external_eval=pd.concat(parts,ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
print(external_eval.result.value_counts().sort_index()); print("Unique hosts:",external_eval.hostname.nunique())

In [ ]:
records=[]
for i,url in enumerate(external_eval.url):
    records.append(extract_deployment_features(url))
    if (i+1)%2000==0: print("Processed",i+1,"/",len(external_eval))
X_external=pd.DataFrame(records)[final_deployment_features]; y_external=external_eval.result.astype(int).to_numpy()
external_prob=final_deployment_model.predict_proba(X_external)[:,1]; external_pred=(external_prob>=0.5).astype(int)
print(X_external.shape,"missing=",X_external.isna().sum().sum())

In [ ]:
def evaluate_binary(y_true,pred,prob):
    tn,fp,fn,tp=confusion_matrix(y_true,pred).ravel()
    return {"n":len(y_true),"accuracy":accuracy_score(y_true,pred),"precision":precision_score(y_true,pred,zero_division=0),"recall":recall_score(y_true,pred,zero_division=0),"f1":f1_score(y_true,pred,zero_division=0),"auroc":roc_auc_score(y_true,prob),"mcc":matthews_corrcoef(y_true,pred),"tn":tn,"fp":fp,"fn":fn,"tp":tp}
external_metrics=evaluate_binary(y_external,external_pred,external_prob); external_metrics_df=pd.DataFrame([external_metrics]); display(external_metrics_df.round(4))

In [ ]:
INTERNAL_METRICS={"n":1777,"f1":0.881087,"precision":0.896313,"recall":0.866370,"accuracy":0.881823,"auroc":0.950949,"mcc":0.764140}
comparison_df=pd.DataFrame([{"dataset":"Internal hostname-disjoint holdout",**INTERNAL_METRICS},{"dataset":"External strict unseen-host",**{k:external_metrics[k] for k in ["n","f1","precision","recall","accuracy","auroc","mcc"]}}]); display(comparison_df.round(4))
generalization_gap=pd.DataFrame([{"delta_f1_external_minus_internal":external_metrics["f1"]-INTERNAL_METRICS["f1"],"delta_auroc_external_minus_internal":external_metrics["auroc"]-INTERNAL_METRICS["auroc"],"delta_mcc_external_minus_internal":external_metrics["mcc"]-INTERNAL_METRICS["mcc"]}]); display(generalization_gap.round(4))

In [ ]:
hosts=external_eval.hostname.astype(str).to_numpy(); unique_hosts=np.unique(hosts); host_to_idx={h:np.where(hosts==h)[0] for h in unique_hosts}; rows=[]
for b in range(3000):
    sampled=rng.choice(unique_hosts,size=len(unique_hosts),replace=True); idx=np.concatenate([host_to_idx[h] for h in sampled]); yb=y_external[idx]
    if len(np.unique(yb))<2: continue
    rows.append({"f1":f1_score(yb,external_pred[idx],zero_division=0),"auroc":roc_auc_score(yb,external_prob[idx]),"mcc":matthews_corrcoef(yb,external_pred[idx])})
bootstrap_df=pd.DataFrame(rows)
ci=[]
for m in ["f1","auroc","mcc"]:
    ci.append({"metric":m,"point_estimate":external_metrics[m],"bootstrap_mean":bootstrap_df[m].mean(),"ci_lower":np.percentile(bootstrap_df[m],2.5),"ci_upper":np.percentile(bootstrap_df[m],97.5)})
external_ci_summary=pd.DataFrame(ci); display(external_ci_summary.round(4))

In [ ]:
X_internal=training_feature_data[final_deployment_features].astype(float).copy(); rows=[]
for f in final_deployment_features:
    a=X_internal[f].dropna().to_numpy(); b=X_external[f].dropna().to_numpy(); sa=np.std(a,ddof=1); sb=np.std(b,ddof=1); pooled=np.sqrt((sa**2+sb**2)/2); smd=(np.mean(b)-np.mean(a))/pooled if pooled>0 else np.nan; ks,pk=ks_2samp(a,b)
    rows.append({"feature":f,"internal_mean":np.mean(a),"external_mean":np.mean(b),"internal_median":np.median(a),"external_median":np.median(b),"internal_std":sa,"external_std":sb,"standardized_mean_difference":smd,"abs_smd":abs(smd) if not np.isnan(smd) else np.nan,"ks_statistic":ks,"ks_p_value":pk,"wasserstein_distance":wasserstein_distance(a,b)})
feature_shift_df=pd.DataFrame(rows).sort_values("abs_smd",ascending=False).reset_index(drop=True); display(feature_shift_df.round(4))

In [ ]:
rows=[]; internal_labels=training_feature_data.label.astype(int).to_numpy()
for lab in [0,1]:
    im=internal_labels==lab; em=y_external==lab
    for f in final_deployment_features:
        a=X_internal.loc[im,f].dropna().to_numpy(); b=X_external.loc[em,f].dropna().to_numpy(); sa=np.std(a,ddof=1); sb=np.std(b,ddof=1); pooled=np.sqrt((sa**2+sb**2)/2); smd=(np.mean(b)-np.mean(a))/pooled if pooled>0 else np.nan; ks,pk=ks_2samp(a,b)
        rows.append({"label":lab,"feature":f,"internal_mean":np.mean(a),"external_mean":np.mean(b),"smd":smd,"abs_smd":abs(smd) if not np.isnan(smd) else np.nan,"ks_statistic":ks,"ks_p_value":pk})
class_shift_df=pd.DataFrame(rows).sort_values(["label","abs_smd"],ascending=[True,False]); display(class_shift_df.round(4))

In [ ]:
shift_summary=pd.DataFrame([{"features_abs_smd_ge_0.2":int((feature_shift_df.abs_smd>=0.2).sum()),"features_abs_smd_ge_0.5":int((feature_shift_df.abs_smd>=0.5).sum()),"features_abs_smd_ge_0.8":int((feature_shift_df.abs_smd>=0.8).sum()),"largest_shift_feature":feature_shift_df.iloc[0].feature,"largest_abs_smd":feature_shift_df.iloc[0].abs_smd,"mean_abs_smd":feature_shift_df.abs_smd.mean()}]); display(shift_summary.round(4))
top=feature_shift_df.head(10).sort_values("abs_smd"); fig,ax=plt.subplots(figsize=(8,5)); ax.barh(top.feature,top.abs_smd); ax.set_xlabel("Absolute standardized mean difference"); ax.set_title("Largest Internal–External Feature Shifts"); plt.tight_layout(); plt.show()

In [ ]:
if semantics_verified:
    pm=EXTERNAL_DATASET_METADATA["positive_class_meaning"].lower(); desc="external phishing-URL validation" if "phish" in pm else "cross-domain malicious-URL generalization stress test"
else:
    desc="external URL-domain generalization stress test with unverified label semantics"
claim_safety_df=pd.DataFrame([{"semantics_verified":semantics_verified,"recommended_description":desc,"model_retrained_on_external":False,"feature_selection_on_external":False,"threshold_tuned_on_external":False,"external_role":"frozen-model out-of-domain evaluation only"}]); display(claim_safety_df)

In [ ]:
external_prediction_df=external_eval[["url","hostname","result"]].copy(); external_prediction_df["predicted_label"]=external_pred; external_prediction_df["predicted_probability"]=external_prob; external_prediction_df["correct"]=external_prediction_df.result.to_numpy()==external_pred
paper_ready_summary=pd.DataFrame([{"Evaluation":"Internal hostname-disjoint","N":INTERNAL_METRICS["n"],"F1":INTERNAL_METRICS["f1"],"AUROC":INTERNAL_METRICS["auroc"],"MCC":INTERNAL_METRICS["mcc"],"Entity_overlap_control":"Hostname-disjoint"},{"Evaluation":"External strict unseen-host","N":external_metrics["n"],"F1":external_metrics["f1"],"AUROC":external_metrics["auroc"],"MCC":external_metrics["mcc"],"Entity_overlap_control":"No exact URL/hostname overlap"}]); display(paper_ready_summary.round(4))

In [ ]:
metadata_df.to_csv(OUT/"external_dataset_metadata_audit.csv",index=False); audit_summary.to_csv(OUT/"external_basic_audit.csv",index=False); deployment_audit_df.to_csv(OUT/"deployment_consistency_mismatches.csv",index=False); overlap_summary.to_csv(OUT/"external_overlap_summary.csv",index=False); unseen_summary.to_csv(OUT/"external_unseen_host_summary.csv",index=False); external_metrics_df.to_csv(OUT/"external_metrics.csv",index=False); comparison_df.to_csv(OUT/"internal_external_comparison.csv",index=False); generalization_gap.to_csv(OUT/"generalization_gap.csv",index=False); bootstrap_df.to_csv(OUT/"external_hostname_bootstrap_samples.csv",index=False); external_ci_summary.to_csv(OUT/"external_hostname_bootstrap_ci.csv",index=False); feature_shift_df.to_csv(OUT/"overall_feature_shift.csv",index=False); class_shift_df.to_csv(OUT/"class_conditional_feature_shift.csv",index=False); shift_summary.to_csv(OUT/"feature_shift_summary.csv",index=False); claim_safety_df.to_csv(OUT/"claim_safety_summary.csv",index=False); external_prediction_df.to_csv(OUT/"external_predictions.csv",index=False); paper_ready_summary.to_csv(OUT/"paper_ready_external_summary.csv",index=False)
print("Saved to:",OUT.resolve()); [print("-",p.name) for p in OUT.iterdir()]

## Interpretation

The frozen 10-feature lexical model showed strong performance on the internal hostname-disjoint test set, with an AUROC of 0.9509 and F1-score of 0.8811. However, performance dropped substantially on the strict unseen-host external dataset, where AUROC decreased to 0.5384 and F1-score to 0.6110. The hostname-grouped bootstrap confidence intervals confirmed that this degradation was stable rather than caused by sampling variation.

The domain-shift analysis also showed substantial changes in several important lexical features, particularly `special_char_ratio`, `num_subdomains`, `suspicious_keyword_count`, `num_dots`, and `num_hyphens`. This indicates that the statistical patterns learned from the primary phishing-oriented dataset do not transfer reliably to the broader external malicious-URL distribution.

Therefore, Experiment 3 provides evidence of a clear **cross-domain generalization gap**. Strong in-domain performance, even under hostname-disjoint evaluation, does not necessarily imply reliable performance when the underlying URL distribution changes. Since the external positive class represents broader malicious URLs rather than phishing exclusively, this experiment should be interpreted as a **cross-domain malicious-URL stress test**, not as direct external quishing validation.

# Experiment 4 — Real-World QR Distortion and End-to-End Robustness
## Objective
This experiment evaluates the complete deployment chain:
\[
QR\ Image
\rightarrow
Distortion
\rightarrow
QR\ Decoding
\rightarrow
10\ Lexical\ Features
\rightarrow
Frozen\ XGBoost
\rightarrow
Prediction
\]
The goal is to determine where performance degrades under realistic QR-image distortions.
Three levels are separated:
1. Decoding robustness
   - Does the QR decoder return a payload?
   - Is the decoded URL still correct?
2. Classifier robustness after decoding
   - If a URL is successfully decoded, does the frozen lexical classifier still perform well?
3. Strict end-to-end robustness
   - What proportion of all QR samples are both correctly decoded and correctly classified?
The classifier is never retrained or tuned on distorted samples.
## Distortions
- Clean
- Gaussian blur: σ = 0.5, 1.0, 1.5
- Rotation: 5°, 10°, 15°
- JPEG compression: quality = 80, 50, 30
- Low resolution: scale = 0.75, 0.50, 0.25
This directly tests the main operational limitation of a lexical detector: it requires successful payload extraction before classification.

In [ ]:
import cv2
import json
import joblib
import pickle
import warnings
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

MAPPING_CANDIDATES = [
    Path("experiment1_rawpixel_hostname_disjoint_results") / "mapped_clean_samples.csv",
    Path("qr_full_feature_dataset_v2.csv"),
    Path("experiment2_full_feature_dataset.csv")
]

MODEL_PATH = Path("FINAL_10feature_lightweight_xgb.pkl")
CONFIG_PATH = Path("FINAL_10feature_deployment_config.json")

OUT = Path("experiment4_realworld_qr_robustness")
OUT.mkdir(exist_ok=True)

print("Ready.")

In [ ]:
with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f)).astype(int)

mapping_path = None

for candidate in MAPPING_CANDIDATES:
    if candidate.exists():
        mapping_path = candidate
        break

if mapping_path is None:
    raise FileNotFoundError(
        "Could not find the clean URL mapping from Experiment 1/2."
    )

mapping_df = pd.read_csv(mapping_path)

required = ["original_index", "url", "hostname", "label"]
missing = [c for c in required if c not in mapping_df.columns]

if missing:
    raise ValueError(
        f"Mapping file is missing required columns: {missing}"
    )

mapping_df = mapping_df.dropna(
    subset=["original_index", "url", "hostname", "label"]
).copy()

mapping_df["original_index"] = mapping_df["original_index"].astype(int)
mapping_df["label"] = mapping_df["label"].astype(int)

print("QR array:", qr_codes.shape)
print("Mapped clean samples:", len(mapping_df))
print("Class counts:")
print(mapping_df["label"].value_counts())
print("Unique hostnames:", mapping_df["hostname"].nunique())

mapped_labels = labels[mapping_df["original_index"].to_numpy()]
print("Label mismatches:", np.sum(mapped_labels != mapping_df["label"].to_numpy()))

assert np.sum(mapped_labels != mapping_df["label"].to_numpy()) == 0

In [ ]:
USE_FULL_DATASET = True
USE_FULL_DATASET = False
N_PER_CLASS = 1000

if USE_FULL_DATASET:

    eval_df = (
        mapping_df
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )

else:

    parts = []

    for label_value in [0, 1]:

        subset = mapping_df[
            mapping_df["label"] == label_value
        ]

        n_take = min(
            N_PER_CLASS,
            len(subset)
        )

        parts.append(
            subset.sample(
                n=n_take,
                random_state=SEED + label_value
            )
        )

    eval_df = (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )

print("Evaluation N:", len(eval_df))
print(eval_df["label"].value_counts().sort_index())
print("Unique hostnames:", eval_df["hostname"].nunique())

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Frozen model not found: {MODEL_PATH.resolve()}"
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Deployment config not found: {CONFIG_PATH.resolve()}"
    )

model = joblib.load(MODEL_PATH)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    deployment_config = json.load(f)

FINAL_FEATURES = list(
    deployment_config["features"]
)

print("Frozen features:")
print(FINAL_FEATURES)

print("Decision threshold: 0.5")

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def ensure_scheme(url):

    url = str(url).strip()

    if not url.lower().startswith(
        ("http://", "https://")
    ):
        return "http://" + url

    return url


def count_subdomains(hostname):

    hostname = str(hostname).strip().lower()

    if hostname.startswith("www."):
        hostname = hostname[4:]

    parts = [
        p
        for p in hostname.split(".")
        if p
    ]

    return max(
        len(parts) - 2,
        0
    )


def extract_deployment_features(url):

    raw_url = str(url).strip()
    lower_url = raw_url.lower()

    parsed = urlsplit(
        ensure_scheme(raw_url)
    )

    hostname = (
        parsed.hostname
        or ""
    ).lower()

    path = parsed.path or ""
    query = parsed.query or ""

    url_length = len(raw_url)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    return {
        "num_subdomains":
            count_subdomains(hostname),

        "suspicious_keyword_count":
            sum(
                k in lower_url
                for k in SUSPICIOUS_KEYWORDS
            ),

        "num_digits":
            num_digits,

        "path_length":
            len(path),

        "hostname_length":
            len(hostname),

        "url_length":
            url_length,

        "num_dots":
            raw_url.count("."),

        "special_char_ratio":
            num_special_chars / max(url_length, 1),

        "num_hyphens":
            raw_url.count("-"),

        "query_length":
            len(query)
    }


assert set(
    extract_deployment_features(
        "https://example.com/a?x=1"
    ).keys()
) == set(FINAL_FEATURES)

print("Feature extractor verified.")

In [ ]:
def normalize_url_for_comparison(url):

    if url is None:
        return None

    url = str(url).strip()

    if url == "":
        return None

    parse_url = ensure_scheme(url)

    try:
        parsed = urlsplit(parse_url)
    except Exception:
        return url.lower()

    hostname = (
        parsed.hostname
        or ""
    ).lower()

    if hostname.startswith("www."):
        hostname = hostname[4:]

    try:
        port = parsed.port
    except Exception:
        port = None

    netloc = hostname

    if port is not None:
        netloc += f":{port}"

    return urlunsplit((
        parsed.scheme.lower(),
        netloc,
        parsed.path or "",
        parsed.query or "",
        ""
    ))

In [ ]:
MODULE_SCALE = 10
QUIET_ZONE_MODULES = 4


def render_qr(qr_matrix):

    qr = np.asarray(qr_matrix)

    # IMPORTANT:
    # In the Trad & Chehab dataset:
    # 0 = black module
    # 1 = white module
    #
    # Therefore direct multiplication gives:
    # 0 -> 0 (black)
    # 1 -> 255 (white)

    if qr.max() <= 1:
        img = (qr * 255).astype(np.uint8)
    else:
        img = qr.astype(np.uint8)

    # Add 4-module white quiet zone BEFORE upscaling
    img = cv2.copyMakeBorder(
        img,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        cv2.BORDER_CONSTANT,
        value=255
    )

    # Upscale without smoothing the QR modules
    img = cv2.resize(
        img,
        None,
        fx=MODULE_SCALE,
        fy=MODULE_SCALE,
        interpolation=cv2.INTER_NEAREST
    )

    return img

In [ ]:
def apply_gaussian_blur(img, sigma):

    k = max(
        3,
        int(
            2 * round(3 * sigma) + 1
        )
    )

    if k % 2 == 0:
        k += 1

    return cv2.GaussianBlur(
        img,
        (k, k),
        sigmaX=sigma,
        sigmaY=sigma
    )


def rotate_bound_white(img, angle):

    h, w = img.shape[:2]

    center = (
        w / 2,
        h / 2
    )

    M = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int(
        h * sin
        +
        w * cos
    )

    new_h = int(
        h * cos
        +
        w * sin
    )

    M[0, 2] += (
        new_w / 2
        -
        center[0]
    )

    M[1, 2] += (
        new_h / 2
        -
        center[1]
    )

    return cv2.warpAffine(
        img,
        M,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255
    )


def apply_jpeg(img, quality):

    encode_param = [
        int(
            cv2.IMWRITE_JPEG_QUALITY
        ),
        int(quality)
    ]

    ok, encoded = cv2.imencode(
        ".jpg",
        img,
        encode_param
    )

    if not ok:
        raise RuntimeError(
            "JPEG encoding failed"
        )

    decoded = cv2.imdecode(
        encoded,
        cv2.IMREAD_GRAYSCALE
    )

    return decoded


def apply_low_resolution(
    img,
    scale
):

    h, w = img.shape[:2]

    small_w = max(
        8,
        int(
            round(w * scale)
        )
    )

    small_h = max(
        8,
        int(
            round(h * scale)
        )
    )

    small = cv2.resize(
        img,
        (small_w, small_h),
        interpolation=cv2.INTER_AREA
    )

    restored = cv2.resize(
        small,
        (w, h),
        interpolation=cv2.INTER_LINEAR
    )

    return restored


def apply_condition(
    img,
    kind,
    severity
):

    if kind == "clean":
        return img.copy()

    if kind == "blur":
        return apply_gaussian_blur(
            img,
            sigma=float(severity)
        )

    if kind == "rotation":
        return rotate_bound_white(
            img,
            angle=float(severity)
        )

    if kind == "jpeg":
        return apply_jpeg(
            img,
            quality=int(severity)
        )

    if kind == "scale":
        return apply_low_resolution(
            img,
            scale=float(severity)
        )

    raise ValueError(
        f"Unknown condition: {kind}"
    )

In [ ]:
CONDITIONS = [
    ("clean", 0),

    ("blur", 0.5),
    ("blur", 1.0),
    ("blur", 1.5),

    ("rotation", 5),
    ("rotation", 10),
    ("rotation", 15),

    ("jpeg", 80),
    ("jpeg", 50),
    ("jpeg", 30),

    ("scale", 0.75),
    ("scale", 0.50),
    ("scale", 0.25)
]


def condition_name(kind, severity):

    if kind == "clean":
        return "Clean"

    if kind == "blur":
        return f"Blur_sigma_{severity}"

    if kind == "rotation":
        return f"Rotation_{severity}deg"

    if kind == "jpeg":
        return f"JPEG_Q{severity}"

    if kind == "scale":
        return f"Scale_{severity}"

    return f"{kind}_{severity}"


condition_table = pd.DataFrame([
    {
        "kind": k,
        "severity": s,
        "condition": condition_name(k, s)
    }
    for k, s in CONDITIONS
])

display(condition_table)

In [ ]:
detector = cv2.QRCodeDetector()


def decode_qr(img):

    data, points, _ = detector.detectAndDecode(
        img
    )

    if data is None:
        return None

    data = str(data).strip()

    if data == "":
        return None

    return data

In [ ]:
records = []

total_jobs = (
    len(eval_df)
    *
    len(CONDITIONS)
)

job_counter = 0

for condition_kind, severity in CONDITIONS:

    cname = condition_name(
        condition_kind,
        severity
    )

    print("\n" + "=" * 80)
    print(cname)

    for row_idx, row in eval_df.iterrows():

        original_index = int(
            row["original_index"]
        )

        true_url = str(
            row["url"]
        )

        true_label = int(
            row["label"]
        )

        hostname = str(
            row["hostname"]
        )

        qr_matrix = qr_codes[
            original_index
        ]

        base_img = render_qr(
            qr_matrix
        )

        distorted_img = apply_condition(
            base_img,
            condition_kind,
            severity
        )

        decoded_url = decode_qr(
            distorted_img
        )

        decode_output_success = (
            decoded_url is not None
        )

        correct_decode = False
        pred = None
        prob = None
        classifier_correct = False

        if decode_output_success:

            correct_decode = (
                normalize_url_for_comparison(
                    decoded_url
                )
                ==
                normalize_url_for_comparison(
                    true_url
                )
            )

            feat = (
                extract_deployment_features(
                    decoded_url
                )
            )

            X_one = pd.DataFrame(
                [feat]
            )[FINAL_FEATURES]

            prob = float(
                model.predict_proba(
                    X_one
                )[0, 1]
            )

            pred = int(
                prob >= 0.5
            )

            classifier_correct = (
                pred == true_label
            )

        if not decode_output_success:

            failure_mode = "no_decode"

        elif not correct_decode:

            failure_mode = "decode_mismatch"

        elif not classifier_correct:

            failure_mode = (
                "classifier_error_after_correct_decode"
            )

        else:

            failure_mode = "success"

        records.append({
            "condition":
                cname,

            "kind":
                condition_kind,

            "severity":
                severity,

            "sample_row":
                row_idx,

            "original_index":
                original_index,

            "hostname":
                hostname,

            "true_label":
                true_label,

            "true_url":
                true_url,

            "decoded_url":
                decoded_url,

            "decode_output_success":
                decode_output_success,

            "correct_decode":
                correct_decode,

            "predicted_label":
                pred,

            "predicted_probability":
                prob,

            "classifier_correct":
                classifier_correct,

            "failure_mode":
                failure_mode
        })

        job_counter += 1

        if job_counter % 2000 == 0:
            print(
                f"Processed {job_counter}/{total_jobs}"
            )


results_df = pd.DataFrame(
    records
)

print("\nDone.")
print("Rows:", len(results_df))

In [ ]:
def safe_auc(y_true, prob):

    if len(
        np.unique(y_true)
    ) < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob
    )


summary_rows = []

for cname, temp in results_df.groupby(
    "condition",
    sort=False
):

    n_total = len(temp)

    decode_output_rate = (
        temp[
            "decode_output_success"
        ].mean()
    )

    correct_decode_rate = (
        temp[
            "correct_decode"
        ].mean()
    )

    decode_mismatch_rate = (
        (
            temp[
                "decode_output_success"
            ]
            &
            ~temp[
                "correct_decode"
            ]
        ).mean()
    )

    correct_decoded = temp[
        temp[
            "correct_decode"
        ]
    ].copy()

    if len(correct_decoded) > 0:

        yt = (
            correct_decoded[
                "true_label"
            ]
            .astype(int)
            .to_numpy()
        )

        yp = (
            correct_decoded[
                "predicted_label"
            ]
            .astype(int)
            .to_numpy()
        )

        pr = (
            correct_decoded[
                "predicted_probability"
            ]
            .astype(float)
            .to_numpy()
        )

        conditional_accuracy = (
            accuracy_score(
                yt,
                yp
            )
        )

        conditional_precision = (
            precision_score(
                yt,
                yp,
                zero_division=0
            )
        )

        conditional_recall = (
            recall_score(
                yt,
                yp,
                zero_division=0
            )
        )

        conditional_f1 = (
            f1_score(
                yt,
                yp,
                zero_division=0
            )
        )

        conditional_auc = (
            safe_auc(
                yt,
                pr
            )
        )

        conditional_mcc = (
            matthews_corrcoef(
                yt,
                yp
            )
        )

    else:

        conditional_accuracy = np.nan
        conditional_precision = np.nan
        conditional_recall = np.nan
        conditional_f1 = np.nan
        conditional_auc = np.nan
        conditional_mcc = np.nan

    strict_e2e_correct = (
        (
            temp[
                "correct_decode"
            ]
            &
            temp[
                "classifier_correct"
            ]
        ).mean()
    )

    phishing_mask = (
        temp[
            "true_label"
        ] == 1
    )

    if phishing_mask.sum() > 0:

        end_to_end_phishing_capture = (
            (
                temp.loc[
                    phishing_mask,
                    "correct_decode"
                ]
                &
                (
                    temp.loc[
                        phishing_mask,
                        "predicted_label"
                    ]
                    ==
                    1
                )
            ).mean()
        )

    else:
        end_to_end_phishing_capture = np.nan

    summary_rows.append({
        "condition":
            cname,

        "kind":
            temp["kind"].iloc[0],

        "severity":
            temp["severity"].iloc[0],

        "n":
            n_total,

        "decode_output_rate":
            decode_output_rate,

        "correct_decode_rate":
            correct_decode_rate,

        "decode_mismatch_rate":
            decode_mismatch_rate,

        "conditional_accuracy":
            conditional_accuracy,

        "conditional_precision":
            conditional_precision,

        "conditional_recall":
            conditional_recall,

        "conditional_f1":
            conditional_f1,

        "conditional_auc":
            conditional_auc,

        "conditional_mcc":
            conditional_mcc,

        "strict_e2e_correct_rate":
            strict_e2e_correct,

        "end_to_end_phishing_capture":
            end_to_end_phishing_capture
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df.round(4)
)

In [ ]:
clean_row = (
    summary_df[
        summary_df[
            "condition"
        ] == "Clean"
    ]
    .iloc[0]
)

retention_df = (
    summary_df.copy()
)

for metric in [
    "decode_output_rate",
    "correct_decode_rate",
    "conditional_f1",
    "conditional_auc",
    "conditional_mcc",
    "strict_e2e_correct_rate",
    "end_to_end_phishing_capture"
]:

    retention_df[
        f"delta_{metric}_vs_clean"
    ] = (
        retention_df[
            metric
        ]
        -
        clean_row[
            metric
        ]
    )

display(
    retention_df.round(4)
)

In [ ]:
failure_table = (
    results_df
    .groupby(
        [
            "condition",
            "failure_mode"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

failure_rate_table = (
    failure_table
    .div(
        failure_table.sum(axis=1),
        axis=0
    )
)

display(
    failure_rate_table.round(4)
)

In [ ]:
plot_order = summary_df[
    "condition"
].tolist()

x = np.arange(
    len(plot_order)
)

fig, ax = plt.subplots(
    figsize=(12, 5)
)

ax.plot(
    x,
    summary_df[
        "correct_decode_rate"
    ],
    marker="o",
    label="Correct decode rate"
)

ax.plot(
    x,
    summary_df[
        "decode_output_rate"
    ],
    marker="o",
    label="Any decode output"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_order,
    rotation=60,
    ha="right"
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Rate"
)

ax.set_title(
    "QR Decoding Robustness"
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 5)
)

ax.plot(
    x,
    summary_df[
        "conditional_auc"
    ],
    marker="o",
    label="Conditional AUROC"
)

ax.plot(
    x,
    summary_df[
        "conditional_f1"
    ],
    marker="o",
    label="Conditional F1"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_order,
    rotation=60,
    ha="right"
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Metric"
)

ax.set_title(
    "Classifier Performance Given Correct Decoding"
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 5)
)

ax.plot(
    x,
    summary_df[
        "strict_e2e_correct_rate"
    ],
    marker="o",
    label="Strict end-to-end correct rate"
)

ax.plot(
    x,
    summary_df[
        "end_to_end_phishing_capture"
    ],
    marker="o",
    label="End-to-end phishing capture"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_order,
    rotation=60,
    ha="right"
)

ax.set_ylim(
    0,
    1.05
)

ax.set_ylabel(
    "Rate"
)

ax.set_title(
    "End-to-End QR-to-Decision Robustness"
)

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
N_BOOT = 1000

bootstrap_summary_rows = []

for cname, temp in results_df.groupby(
    "condition",
    sort=False
):

    temp = temp.reset_index(drop=True)

    hosts = (
        temp["hostname"]
        .astype(str)
        .unique()
    )

    host_to_idx = {
        h:
        np.where(
            temp[
                "hostname"
            ].astype(str).to_numpy()
            ==
            h
        )[0]
        for h in hosts
    }

    boot_records = []

    local_rng = np.random.default_rng(
        SEED
    )

    for b in range(
        N_BOOT
    ):

        sampled_hosts = (
            local_rng.choice(
                hosts,
                size=len(hosts),
                replace=True
            )
        )

        idx = np.concatenate(
            [
                host_to_idx[h]
                for h in sampled_hosts
            ]
        )

        bt = temp.iloc[
            idx
        ]

        correct_decode_rate = (
            bt[
                "correct_decode"
            ].mean()
        )

        strict_e2e = (
            (
                bt[
                    "correct_decode"
                ]
                &
                bt[
                    "classifier_correct"
                ]
            ).mean()
        )

        cd = bt[
            bt[
                "correct_decode"
            ]
        ]

        if (
            len(cd) > 0
            and
            cd[
                "true_label"
            ].nunique() == 2
        ):

            yb = (
                cd[
                    "true_label"
                ]
                .astype(int)
                .to_numpy()
            )

            pb = (
                cd[
                    "predicted_probability"
                ]
                .astype(float)
                .to_numpy()
            )

            pred_b = (
                cd[
                    "predicted_label"
                ]
                .astype(int)
                .to_numpy()
            )

            auc_b = (
                roc_auc_score(
                    yb,
                    pb
                )
            )

            f1_b = (
                f1_score(
                    yb,
                    pred_b,
                    zero_division=0
                )
            )

        else:

            auc_b = np.nan
            f1_b = np.nan

        boot_records.append({
            "correct_decode_rate":
                correct_decode_rate,

            "conditional_auc":
                auc_b,

            "conditional_f1":
                f1_b,

            "strict_e2e_correct_rate":
                strict_e2e
        })

    boot_df = pd.DataFrame(
        boot_records
    )

    for metric in [
        "correct_decode_rate",
        "conditional_auc",
        "conditional_f1",
        "strict_e2e_correct_rate"
    ]:

        vals = (
            boot_df[
                metric
            ]
            .dropna()
            .to_numpy()
        )

        bootstrap_summary_rows.append({
            "condition":
                cname,

            "metric":
                metric,

            "bootstrap_mean":
                np.mean(vals),

            "ci_lower":
                np.percentile(
                    vals,
                    2.5
                ),

            "ci_upper":
                np.percentile(
                    vals,
                    97.5
                )
        })


bootstrap_ci_df = pd.DataFrame(
    bootstrap_summary_rows
)

display(
    bootstrap_ci_df.round(4)
)

In [ ]:
severity_summary = (
    summary_df[
        summary_df[
            "kind"
        ] != "clean"
    ]
    [
        [
            "kind",
            "severity",
            "correct_decode_rate",
            "conditional_f1",
            "conditional_auc",
            "strict_e2e_correct_rate",
            "end_to_end_phishing_capture"
        ]
    ]
    .copy()
)

display(
    severity_summary.round(4)
)

In [ ]:
bottleneck_rows = []

for _, row in summary_df.iterrows():

    decode_loss = (
        clean_row[
            "correct_decode_rate"
        ]
        -
        row[
            "correct_decode_rate"
        ]
    )

    classifier_loss = (
        clean_row[
            "conditional_f1"
        ]
        -
        row[
            "conditional_f1"
        ]
    )

    if (
        abs(decode_loss)
        >
        abs(classifier_loss)
    ):
        dominant = "Decoding"
    else:
        dominant = "Classifier"

    bottleneck_rows.append({
        "condition":
            row[
                "condition"
            ],

        "decode_rate_loss_vs_clean":
            decode_loss,

        "conditional_f1_loss_vs_clean":
            classifier_loss,

        "dominant_degradation_component":
            dominant
    })

bottleneck_df = pd.DataFrame(
    bottleneck_rows
)

display(
    bottleneck_df.round(4)
)

In [ ]:
results_df.to_csv(
    OUT / "sample_level_robustness_results.csv",
    index=False
)

summary_df.to_csv(
    OUT / "condition_level_summary.csv",
    index=False
)

retention_df.to_csv(
    OUT / "performance_retention_vs_clean.csv",
    index=False
)

failure_table.to_csv(
    OUT / "failure_mode_counts.csv"
)

failure_rate_table.to_csv(
    OUT / "failure_mode_rates.csv"
)

bootstrap_ci_df.to_csv(
    OUT / "hostname_grouped_bootstrap_ci.csv",
    index=False
)

severity_summary.to_csv(
    OUT / "severity_summary.csv",
    index=False
)

bottleneck_df.to_csv(
    OUT / "bottleneck_analysis.csv",
    index=False
)

eval_df[
    [
        "original_index",
        "url",
        "hostname",
        "label"
    ]
].to_csv(
    OUT / "evaluation_sample.csv",
    index=False
)

print(
    "Saved to:",
    OUT.resolve()
)

for p in OUT.iterdir():
    print(
        "-",
        p.name
    )

# Interpretation
This experiment separates image-acquisition robustness from classifier robustness.
The key comparison is between correct decode rate and conditional classifier performance. If QR distortions sharply reduce decoding success while AUROC/F1 remain relatively stable among correctly decoded samples, the main operational bottleneck lies in payload extraction rather than the lexical classifier itself.
Conversely, if decoding remains reliable but conditional classification performance falls, the distortion is affecting the decoded representation or downstream decision process.
The strict end-to-end correct rate summarizes the practical reliability of the complete QR-to-decision pipeline. This experiment therefore clarifies the main trade-off of lexical quishing detection: lexical features can provide strong and lightweight predictions after successful decoding, but real-world robustness ultimately depends on the reliability of QR payload extraction.

# Experiment 4 V2 — Real-World QR Distortion and End-to-End Robustness
## Purpose
This is the corrected final version of Experiment 4.
The complete pipeline is:
\[
QR\ Image
\rightarrow
Distortion
\rightarrow
QR\ Decoding
\rightarrow
Verified\ 10\text{-Feature\ Extractor}
\rightarrow
Frozen\ XGBoost
\rightarrow
Prediction
\]
The feature extraction and deployment logic below are copied from the final verified deployment section of the original notebook.
## Mandatory sanity gate
Before any robustness result is accepted, the notebook verifies on the same evaluation samples that:
1. the clean QR image decodes correctly;
2. features extracted from the original URL and clean-decoded URL are identical;
3. frozen-model probabilities and predictions are identical.
This is a stronger validation than comparing against a previous holdout score, because the evaluation sample used in this experiment may differ from the original held-out test set.
## Distortions
- Clean
- Gaussian blur: σ = 0.5, 1.0, 1.5
- Rotation: -15°, -10°, -5°, +5°, +10°, +15°
- JPEG compression: Q = 80, 50, 30
- Low resolution: scale = 0.75, 0.50, 0.25
Both rotation directions are evaluated to avoid conclusions based on a single fixed-angle artifact.

In [ ]:
import re
import cv2
import json
import joblib
import pickle
import warnings
from pathlib import Path
from urllib.parse import urlparse, urlunparse, urlsplit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path(r"C:\\Users\\LENOVO\\Desktop\\FYP\\QuishingDataset")

QR_FILE = DATA_DIR / "qr_codes_29.pickle"
LABEL_FILE = DATA_DIR / "qr_codes_29_labels.pickle"

# Exact artifact names from the original final deployment notebook
FINAL_MODEL_PATH = Path("FINAL_10feature_XGB_for_deployment.pkl")
FINAL_CONFIG_PATH = Path("phase14B_final_robust_features.json")
DATA_PATH = Path("qr_full_feature_dataset_v2.csv")

OUT = Path("experiment4_v2_realworld_qr_robustness")
OUT.mkdir(exist_ok=True)

print("Model:", FINAL_MODEL_PATH.resolve())
print("Config:", FINAL_CONFIG_PATH.resolve())
print("Feature data:", DATA_PATH.resolve())

In [ ]:
with open(QR_FILE, "rb") as f:
    qr_codes = np.asarray(pickle.load(f))

with open(LABEL_FILE, "rb") as f:
    labels = np.asarray(pickle.load(f)).astype(int)

final_deployment_model = joblib.load(
    FINAL_MODEL_PATH
)

with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    deployment_config = json.load(f)

final_deployment_features = list(
    deployment_config["features"]
)

training_feature_data = pd.read_csv(
    DATA_PATH
)

required_cols = [
    "original_index",
    "url",
    "hostname",
    "label"
] + final_deployment_features

missing = [
    c for c in required_cols
    if c not in training_feature_data.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("QR shape:", qr_codes.shape)
print("Feature data shape:", training_feature_data.shape)
print("Final deployment features:")
for i, f in enumerate(final_deployment_features, 1):
    print(i, f)

In [ ]:
def normalize_url(url):
    url = str(url).strip()

    parse_url = url

    if not url.lower().startswith(
        ("http://", "https://")
    ):
        parse_url = "http://" + url

    parsed = urlparse(parse_url)

    hostname = (
        parsed.hostname.lower()
        if parsed.hostname
        else ""
    )

    if hostname.startswith("www."):
        hostname = hostname[4:]

    try:
        port = parsed.port
    except ValueError:
        port = None

    netloc = hostname

    if port is not None:
        netloc += f":{port}"

    scheme = parsed.scheme.lower()

    normalized = urlunparse(
        (
            scheme,
            netloc,
            parsed.path,
            "",
            parsed.query,
            ""
        )
    )

    return normalized

In [ ]:
def ensure_scheme(url):

    url = str(url).strip()

    if not url:
        return ""

    has_real_scheme = re.match(
        r'^[A-Za-z][A-Za-z0-9+.\-]*://',
        url
    )

    if has_real_scheme:
        return url

    return "http://" + url


try:
    import tldextract

    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

except Exception:
    domain_extractor = None


def get_registered_domain(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return ""

    # Exact final compatibility fix from original notebook
    if hostname == "fed.us" or hostname.endswith(".fed.us"):
        return "fed.us"

    if domain_extractor is None:
        return hostname

    ext = domain_extractor(hostname)

    if not ext.domain:
        return hostname

    if ext.suffix:
        return ext.domain + "." + ext.suffix

    return ext.domain


def count_subdomains(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return 0

    registered_domain = get_registered_domain(
        hostname
    )

    hostname_parts = [
        x for x in hostname.split(".")
        if x
    ]

    domain_parts = [
        x for x in registered_domain.split(".")
        if x
    ]

    return max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )


def parse_deployment_url(url):

    raw_url = str(url).strip()

    normalized_url = normalize_url(
        raw_url
    )

    try:
        raw_parsed = urlsplit(
            ensure_scheme(raw_url)
        )

        raw_hostname = (
            raw_parsed.hostname
            or ""
        ).lower()

    except Exception:
        raw_hostname = ""

    try:
        normalized_parsed = urlsplit(
            normalized_url
        )

        normalized_path = (
            normalized_parsed.path
            or ""
        )

        normalized_query = (
            normalized_parsed.query
            or ""
        )

    except Exception:
        normalized_path = ""
        normalized_query = ""

    return {
        "raw_url": raw_url,
        "normalized_url": normalized_url,
        "raw_hostname": raw_hostname,
        "path": normalized_path,
        "query": normalized_query
    }

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def extract_deployment_features(url):

    parsed = parse_deployment_url(
        url
    )

    raw_url = parsed["raw_url"]
    raw_hostname = parsed["raw_hostname"]
    path = parsed["path"]
    query = parsed["query"]

    url_lower = raw_url.lower()

    url_length = len(raw_url)

    # Exact training-compatible definition:
    # keep "www." in raw hostname length
    hostname_length = len(
        raw_hostname
    )

    # Exact training-compatible definition:
    # normalized path/query
    path_length = len(path)
    query_length = len(query)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_dots = raw_url.count(".")
    num_hyphens = raw_url.count("-")

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    num_subdomains = count_subdomains(
        raw_hostname
    )

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio,

        "num_hyphens":
            num_hyphens,

        "query_length":
            query_length
    }


assert list(
    extract_deployment_features(
        "https://www.example.com/a?x=1"
    ).keys()
) == final_deployment_features

print("Exact final deployment extractor loaded.")

In [ ]:
consistency_records = []

for i, row in (
    training_feature_data
    .reset_index(drop=True)
    .iterrows()
):

    new_features = extract_deployment_features(
        row["url"]
    )

    for feature in final_deployment_features:

        old_value = float(
            row[feature]
        )

        new_value = float(
            new_features[feature]
        )

        consistency_records.append({
            "sample": i,
            "feature": feature,
            "old_value": old_value,
            "new_value": new_value,
            "difference":
                old_value - new_value
        })


deployment_consistency_df = pd.DataFrame(
    consistency_records
)

mismatch_df = deployment_consistency_df[
    deployment_consistency_df[
        "difference"
    ].abs() > 1e-12
].copy()

print("Samples checked:", len(training_feature_data))
print("Feature values checked:", len(deployment_consistency_df))
print(
    "Maximum absolute difference:",
    deployment_consistency_df[
        "difference"
    ].abs().max()
)
print("Mismatch count:", len(mismatch_df))

if len(mismatch_df) != 0:
    display(mismatch_df.head(30))
    raise RuntimeError(
        "STOP: deployment extractor no longer matches training features."
    )

print(
    "PASS: exact deployment consistency audit."
)

In [ ]:
USE_FULL_DATASET = False
N_PER_CLASS = 1000

base_df = (
    training_feature_data[
        [
            "original_index",
            "url",
            "hostname",
            "label"
        ]
        +
        final_deployment_features
    ]
    .dropna(
        subset=[
            "original_index",
            "url",
            "hostname",
            "label"
        ]
    )
    .copy()
)

base_df["original_index"] = (
    base_df["original_index"]
    .astype(int)
)

base_df["label"] = (
    base_df["label"]
    .astype(int)
)

if USE_FULL_DATASET:

    eval_df = (
        base_df
        .sample(
            frac=1,
            random_state=SEED
        )
        .reset_index(drop=True)
    )

else:

    parts = []

    for label_value in [0, 1]:

        subset = base_df[
            base_df["label"] == label_value
        ]

        parts.append(
            subset.sample(
                n=min(
                    N_PER_CLASS,
                    len(subset)
                ),
                random_state=
                    SEED + label_value
            )
        )

    eval_df = (
        pd.concat(
            parts,
            ignore_index=True
        )
        .sample(
            frac=1,
            random_state=SEED
        )
        .reset_index(drop=True)
    )

print("Evaluation N:", len(eval_df))
print(eval_df["label"].value_counts().sort_index())

In [ ]:
direct_feature_records = []

for url in eval_df["url"]:
    direct_feature_records.append(
        extract_deployment_features(url)
    )

X_direct = pd.DataFrame(
    direct_feature_records
)[final_deployment_features]

y_eval = (
    eval_df["label"]
    .astype(int)
    .to_numpy()
)

direct_prob = (
    final_deployment_model
    .predict_proba(
        X_direct
    )[:, 1]
)

direct_pred = (
    direct_prob >= 0.5
).astype(int)


def metric_row(
    y_true,
    pred,
    prob
):

    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                pred
            )
    }


direct_metrics = metric_row(
    y_eval,
    direct_pred,
    direct_prob
)

print("Direct-URL baseline on this exact sample:")
display(
    pd.DataFrame(
        [direct_metrics]
    ).round(4)
)

In [ ]:
MODULE_SCALE = 10
QUIET_ZONE_MODULES = 4


def render_qr(qr_matrix):

    qr = np.asarray(
        qr_matrix
    )

    if qr.max() <= 1:
        # Exact orientation confirmed from original dataset:
        # 0 -> black, 1 -> white
        img = (
            qr * 255
        ).astype(np.uint8)

    else:
        img = qr.astype(
            np.uint8
        )

    img = cv2.copyMakeBorder(
        img,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        QUIET_ZONE_MODULES,
        cv2.BORDER_CONSTANT,
        value=255
    )

    img = cv2.resize(
        img,
        None,
        fx=MODULE_SCALE,
        fy=MODULE_SCALE,
        interpolation=cv2.INTER_NEAREST
    )

    return img


detector = cv2.QRCodeDetector()


def decode_qr(img):

    data, _, _ = (
        detector.detectAndDecode(
            img
        )
    )

    if data is None:
        return None

    data = str(data).strip()

    if data == "":
        return None

    return data


def normalize_for_match(url):

    if url is None:
        return None

    try:
        return normalize_url(
            str(url)
        )
    except Exception:
        return str(url).strip()

In [ ]:
clean_audit_records = []

for i, row in eval_df.iterrows():

    idx = int(
        row["original_index"]
    )

    original_url = str(
        row["url"]
    )

    img = render_qr(
        qr_codes[idx]
    )

    decoded_url = decode_qr(
        img
    )

    decode_success = (
        decoded_url is not None
    )

    correct_decode = (
        decode_success
        and
        normalize_for_match(
            decoded_url
        )
        ==
        normalize_for_match(
            original_url
        )
    )

    direct_features = (
        extract_deployment_features(
            original_url
        )
    )

    if decoded_url is not None:

        decoded_features = (
            extract_deployment_features(
                decoded_url
            )
        )

        feature_max_abs_diff = max(
            abs(
                float(
                    direct_features[f]
                )
                -
                float(
                    decoded_features[f]
                )
            )
            for f in final_deployment_features
        )

        X_decoded_one = pd.DataFrame(
            [decoded_features]
        )[final_deployment_features]

        decoded_prob = float(
            final_deployment_model
            .predict_proba(
                X_decoded_one
            )[0, 1]
        )

        decoded_pred = int(
            decoded_prob >= 0.5
        )

    else:

        feature_max_abs_diff = np.nan
        decoded_prob = np.nan
        decoded_pred = np.nan

    clean_audit_records.append({
        "sample":
            i,

        "decode_success":
            decode_success,

        "correct_decode":
            correct_decode,

        "feature_max_abs_diff":
            feature_max_abs_diff,

        "direct_probability":
            float(
                direct_prob[i]
            ),

        "decoded_probability":
            decoded_prob,

        "probability_abs_diff":
            abs(
                float(
                    direct_prob[i]
                )
                -
                decoded_prob
            )
            if not np.isnan(
                decoded_prob
            )
            else np.nan,

        "direct_prediction":
            int(
                direct_pred[i]
            ),

        "decoded_prediction":
            decoded_pred,

        "prediction_match":
            (
                int(
                    direct_pred[i]
                )
                ==
                decoded_pred
            )
            if not np.isnan(
                decoded_pred
            )
            else False
    })


clean_audit_df = pd.DataFrame(
    clean_audit_records
)

clean_summary = pd.DataFrame([{
    "n":
        len(clean_audit_df),

    "decode_output_rate":
        clean_audit_df[
            "decode_success"
        ].mean(),

    "correct_decode_rate":
        clean_audit_df[
            "correct_decode"
        ].mean(),

    "feature_mismatch_count":
        int(
            (
                clean_audit_df[
                    "feature_max_abs_diff"
                ]
                .fillna(np.inf)
                >
                1e-12
            ).sum()
        ),

    "max_feature_abs_diff":
        clean_audit_df[
            "feature_max_abs_diff"
        ].max(),

    "prediction_mismatch_count":
        int(
            (
                ~clean_audit_df[
                    "prediction_match"
                ]
            ).sum()
        ),

    "max_probability_abs_diff":
        clean_audit_df[
            "probability_abs_diff"
        ].max()
}])

display(
    clean_summary.round(8)
)

if clean_summary.loc[
    0,
    "correct_decode_rate"
] < 0.99:
    raise RuntimeError(
        "STOP: clean QR decoding is unexpectedly low."
    )

if clean_summary.loc[
    0,
    "feature_mismatch_count"
] != 0:
    raise RuntimeError(
        "STOP: clean decoded features do not match direct URL features."
    )

if clean_summary.loc[
    0,
    "prediction_mismatch_count"
] != 0:
    raise RuntimeError(
        "STOP: clean decoded predictions do not match direct URL predictions."
    )

print(
    "PASS: clean QR pipeline exactly matches direct-URL deployment pipeline."
)

In [ ]:
clean_decoded_prob = (
    clean_audit_df[
        "decoded_probability"
    ]
    .astype(float)
    .to_numpy()
)

clean_decoded_pred = (
    clean_audit_df[
        "decoded_prediction"
    ]
    .astype(int)
    .to_numpy()
)

clean_decoded_metrics = (
    metric_row(
        y_eval,
        clean_decoded_pred,
        clean_decoded_prob
    )
)

sanity_metric_comparison = pd.DataFrame([
    {
        "pipeline":
            "Direct URL",
        **direct_metrics
    },
    {
        "pipeline":
            "Clean QR -> decoded URL",
        **clean_decoded_metrics
    }
])

display(
    sanity_metric_comparison.round(6)
)

In [ ]:
def apply_gaussian_blur(
    img,
    sigma
):

    k = max(
        3,
        int(
            2 * round(
                3 * sigma
            ) + 1
        )
    )

    if k % 2 == 0:
        k += 1

    return cv2.GaussianBlur(
        img,
        (k, k),
        sigmaX=sigma,
        sigmaY=sigma
    )


def rotate_bound_white(
    img,
    angle
):

    h, w = img.shape[:2]

    center = (
        w / 2,
        h / 2
    )

    M = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    cos = abs(
        M[0, 0]
    )

    sin = abs(
        M[0, 1]
    )

    new_w = int(
        h * sin
        +
        w * cos
    )

    new_h = int(
        h * cos
        +
        w * sin
    )

    M[0, 2] += (
        new_w / 2
        -
        center[0]
    )

    M[1, 2] += (
        new_h / 2
        -
        center[1]
    )

    return cv2.warpAffine(
        img,
        M,
        (
            new_w,
            new_h
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255
    )


def apply_jpeg(
    img,
    quality
):

    ok, encoded = cv2.imencode(
        ".jpg",
        img,
        [
            int(
                cv2.IMWRITE_JPEG_QUALITY
            ),
            int(quality)
        ]
    )

    if not ok:
        raise RuntimeError(
            "JPEG encoding failed."
        )

    return cv2.imdecode(
        encoded,
        cv2.IMREAD_GRAYSCALE
    )


def apply_low_resolution(
    img,
    scale
):

    h, w = img.shape[:2]

    small = cv2.resize(
        img,
        (
            max(
                8,
                int(
                    round(
                        w * scale
                    )
                )
            ),
            max(
                8,
                int(
                    round(
                        h * scale
                    )
                )
            )
        ),
        interpolation=cv2.INTER_AREA
    )

    return cv2.resize(
        small,
        (
            w,
            h
        ),
        interpolation=cv2.INTER_LINEAR
    )


def apply_condition(
    img,
    kind,
    severity
):

    if kind == "clean":
        return img.copy()

    if kind == "blur":
        return apply_gaussian_blur(
            img,
            float(severity)
        )

    if kind == "rotation":
        return rotate_bound_white(
            img,
            float(severity)
        )

    if kind == "jpeg":
        return apply_jpeg(
            img,
            int(severity)
        )

    if kind == "scale":
        return apply_low_resolution(
            img,
            float(severity)
        )

    raise ValueError(kind)

In [ ]:
CONDITIONS = [
    ("clean", 0),

    ("blur", 0.5),
    ("blur", 1.0),
    ("blur", 1.5),

    ("rotation", -15),
    ("rotation", -10),
    ("rotation", -5),
    ("rotation", 5),
    ("rotation", 10),
    ("rotation", 15),

    ("jpeg", 80),
    ("jpeg", 50),
    ("jpeg", 30),

    ("scale", 0.75),
    ("scale", 0.50),
    ("scale", 0.25)
]


def condition_name(
    kind,
    severity
):

    if kind == "clean":
        return "Clean"

    if kind == "blur":
        return f"Blur_sigma_{severity}"

    if kind == "rotation":
        sign = (
            "plus"
            if float(
                severity
            ) > 0
            else "minus"
        )

        return (
            f"Rotation_{sign}_"
            f"{abs(float(severity))}deg"
        )

    if kind == "jpeg":
        return f"JPEG_Q{severity}"

    if kind == "scale":
        return f"Scale_{severity}"

    return (
        f"{kind}_{severity}"
    )

In [ ]:
records = []

total = (
    len(eval_df)
    *
    len(CONDITIONS)
)

counter = 0

for kind, severity in CONDITIONS:

    cname = condition_name(
        kind,
        severity
    )

    print(
        "\n",
        cname
    )

    for row_idx, row in eval_df.iterrows():

        idx = int(
            row["original_index"]
        )

        original_url = str(
            row["url"]
        )

        true_label = int(
            row["label"]
        )

        hostname = str(
            row["hostname"]
        )

        base_img = render_qr(
            qr_codes[idx]
        )

        distorted_img = apply_condition(
            base_img,
            kind,
            severity
        )

        decoded_url = decode_qr(
            distorted_img
        )

        decode_output_success = (
            decoded_url is not None
        )

        correct_decode = (
            decode_output_success
            and
            normalize_for_match(
                decoded_url
            )
            ==
            normalize_for_match(
                original_url
            )
        )

        pred = np.nan
        prob = np.nan
        classifier_correct = False

        if decode_output_success:

            feature_dict = (
                extract_deployment_features(
                    decoded_url
                )
            )

            X_one = pd.DataFrame(
                [feature_dict]
            )[final_deployment_features]

            prob = float(
                final_deployment_model
                .predict_proba(
                    X_one
                )[0, 1]
            )

            pred = int(
                prob >= 0.5
            )

            classifier_correct = (
                pred == true_label
            )

        if not decode_output_success:

            failure_mode = (
                "no_decode"
            )

        elif not correct_decode:

            failure_mode = (
                "decode_mismatch"
            )

        elif not classifier_correct:

            failure_mode = (
                "classifier_error_after_correct_decode"
            )

        else:

            failure_mode = (
                "success"
            )

        records.append({
            "condition":
                cname,

            "kind":
                kind,

            "severity":
                severity,

            "sample_row":
                row_idx,

            "original_index":
                idx,

            "hostname":
                hostname,

            "true_label":
                true_label,

            "true_url":
                original_url,

            "decoded_url":
                decoded_url,

            "decode_output_success":
                decode_output_success,

            "correct_decode":
                correct_decode,

            "predicted_label":
                pred,

            "predicted_probability":
                prob,

            "classifier_correct":
                classifier_correct,

            "failure_mode":
                failure_mode
        })

        counter += 1

        if counter % 3000 == 0:
            print(
                f"{counter}/{total}"
            )


results_df = pd.DataFrame(
    records
)

print(
    "Rows:",
    len(results_df)
)

In [ ]:
def safe_auc(
    y_true,
    prob
):

    if len(
        np.unique(
            y_true
        )
    ) < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob
    )


summary_rows = []

for cname, temp in (
    results_df.groupby(
        "condition",
        sort=False
    )
):

    n = len(temp)

    correct_decoded = temp[
        temp[
            "correct_decode"
        ]
    ].copy()

    if len(
        correct_decoded
    ) > 0:

        yt = (
            correct_decoded[
                "true_label"
            ]
            .astype(int)
            .to_numpy()
        )

        yp = (
            correct_decoded[
                "predicted_label"
            ]
            .astype(int)
            .to_numpy()
        )

        pr = (
            correct_decoded[
                "predicted_probability"
            ]
            .astype(float)
            .to_numpy()
        )

        cond = metric_row(
            yt,
            yp,
            pr
        )

    else:

        cond = {
            k: np.nan
            for k in [
                "accuracy",
                "precision",
                "recall",
                "f1",
                "auc",
                "mcc"
            ]
        }

    strict_e2e = (
        (
            temp[
                "correct_decode"
            ]
            &
            temp[
                "classifier_correct"
            ]
        ).mean()
    )

    phishing_mask = (
        temp[
            "true_label"
        ] == 1
    )

    e2e_phishing_capture = (
        (
            temp.loc[
                phishing_mask,
                "correct_decode"
            ]
            &
            (
                temp.loc[
                    phishing_mask,
                    "predicted_label"
                ] == 1
            )
        ).mean()
    )

    summary_rows.append({
        "condition":
            cname,

        "kind":
            temp[
                "kind"
            ].iloc[0],

        "severity":
            temp[
                "severity"
            ].iloc[0],

        "n":
            n,

        "decode_output_rate":
            temp[
                "decode_output_success"
            ].mean(),

        "correct_decode_rate":
            temp[
                "correct_decode"
            ].mean(),

        "decode_mismatch_rate":
            (
                temp[
                    "decode_output_success"
                ]
                &
                ~temp[
                    "correct_decode"
                ]
            ).mean(),

        "conditional_accuracy":
            cond["accuracy"],

        "conditional_precision":
            cond["precision"],

        "conditional_recall":
            cond["recall"],

        "conditional_f1":
            cond["f1"],

        "conditional_auc":
            cond["auc"],

        "conditional_mcc":
            cond["mcc"],

        "strict_e2e_correct_rate":
            strict_e2e,

        "end_to_end_phishing_capture":
            e2e_phishing_capture
    })


summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df.round(4)
)

In [ ]:
rotation_rows = summary_df[
    summary_df["kind"] == "rotation"
].copy()

rotation_rows[
    "abs_severity"
] = (
    rotation_rows[
        "severity"
    ]
    .astype(float)
    .abs()
)

rotation_aggregate = (
    rotation_rows
    .groupby(
        "abs_severity"
    )
    .agg(
        mean_correct_decode_rate=(
            "correct_decode_rate",
            "mean"
        ),
        min_correct_decode_rate=(
            "correct_decode_rate",
            "min"
        ),
        max_correct_decode_rate=(
            "correct_decode_rate",
            "max"
        ),
        mean_conditional_f1=(
            "conditional_f1",
            "mean"
        ),
        mean_conditional_auc=(
            "conditional_auc",
            "mean"
        ),
        mean_strict_e2e_correct_rate=(
            "strict_e2e_correct_rate",
            "mean"
        )
    )
    .reset_index()
)

display(
    rotation_aggregate.round(4)
)

In [ ]:
failure_counts = (
    results_df
    .groupby(
        [
            "condition",
            "failure_mode"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

failure_rates = (
    failure_counts
    .div(
        failure_counts.sum(
            axis=1
        ),
        axis=0
    )
)

display(
    failure_rates.round(4)
)

In [ ]:
clean = (
    summary_df[
        summary_df[
            "condition"
        ] == "Clean"
    ]
    .iloc[0]
)

retention_df = (
    summary_df.copy()
)

for metric in [
    "correct_decode_rate",
    "conditional_f1",
    "conditional_auc",
    "conditional_mcc",
    "strict_e2e_correct_rate",
    "end_to_end_phishing_capture"
]:

    retention_df[
        f"delta_{metric}_vs_clean"
    ] = (
        retention_df[
            metric
        ]
        -
        clean[
            metric
        ]
    )

display(
    retention_df.round(4)
)

In [ ]:
bottleneck_rows = []

for _, row in (
    summary_df.iterrows()
):

    decode_loss = (
        clean[
            "correct_decode_rate"
        ]
        -
        row[
            "correct_decode_rate"
        ]
    )

    classifier_loss = (
        clean[
            "conditional_f1"
        ]
        -
        row[
            "conditional_f1"
        ]
    )

    if (
        abs(
            decode_loss
        )
        >
        abs(
            classifier_loss
        )
    ):

        dominant = (
            "Decoding"
        )

    else:

        dominant = (
            "Classifier"
        )

    bottleneck_rows.append({
        "condition":
            row[
                "condition"
            ],

        "decode_rate_loss_vs_clean":
            decode_loss,

        "conditional_f1_loss_vs_clean":
            classifier_loss,

        "dominant_degradation_component":
            dominant
    })


bottleneck_df = pd.DataFrame(
    bottleneck_rows
)

display(
    bottleneck_df.round(4)
)

In [ ]:
clean_summary.to_csv(
    OUT / "clean_pipeline_sanity_summary.csv",
    index=False
)

clean_audit_df.to_csv(
    OUT / "clean_pipeline_sample_audit.csv",
    index=False
)

sanity_metric_comparison.to_csv(
    OUT / "direct_vs_clean_decoded_metrics.csv",
    index=False
)

results_df.to_csv(
    OUT / "sample_level_robustness_results.csv",
    index=False
)

summary_df.to_csv(
    OUT / "condition_level_summary.csv",
    index=False
)

rotation_aggregate.to_csv(
    OUT / "rotation_absolute_severity_summary.csv",
    index=False
)

failure_counts.to_csv(
    OUT / "failure_mode_counts.csv"
)

failure_rates.to_csv(
    OUT / "failure_mode_rates.csv"
)

retention_df.to_csv(
    OUT / "performance_retention_vs_clean.csv",
    index=False
)

bottleneck_df.to_csv(
    OUT / "bottleneck_analysis.csv",
    index=False
)

deployment_consistency_df.to_csv(
    OUT / "full_deployment_consistency_audit.csv",
    index=False
)

print(
    "Saved to:",
    OUT.resolve()
)

for p in OUT.iterdir():
    print(
        "-",
        p.name
    )

## Interpretation

The clean-pipeline sanity check confirmed exact consistency between direct URL evaluation and the complete QR-to-decision pipeline, with zero feature or prediction mismatches.

Across Gaussian blur, JPEG compression, and image downscaling, the conditional classifier performance remained almost unchanged whenever the QR payload was correctly decoded. For example, the clean AUROC of 0.9694 remained approximately 0.969 under blur and JPEG compression and 0.9684 even at 25% image scale. In contrast, end-to-end degradation closely followed reductions in QR decoding success.

Rotation produced a non-monotonic, angle-specific decoding effect, with a pronounced reduction around ±10° but substantially higher decoding rates at ±5° and ±15°. Importantly, conditional classification performance remained high even under these conditions.

Overall, the results indicate that the principal robustness bottleneck of the lexical quishing pipeline lies in **QR payload extraction rather than downstream lexical classification**. Once the original URL is successfully recovered, the frozen lexical classifier is largely insensitive to the tested image distortions.

# Experiment 5 — Deployment Efficiency of the Final Lightweight Detector
## Objective
This final experiment evaluates whether the proposed 10-feature lexical detector is not only accurate, but also lightweight enough for practical deployment.
The experiment measures:
- serialized model size,
- feature extraction latency,
- prediction latency,
- total URL-to-decision latency,
- throughput in URLs per second.
Where the full lexical model is available, the same measurements are repeated for comparison.

The goal is not to claim hardware-independent absolute speed. Instead, the experiment provides a reproducible deployment-efficiency profile on the current machine and quantifies the relative efficiency of the final 10-feature detector.

In [ ]:
import os
import re
import json
import time
import joblib
import warnings
from pathlib import Path
from urllib.parse import urlparse, urlunparse, urlsplit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

DATA_PATH = Path("qr_full_feature_dataset_v2.csv")

FINAL_MODEL_PATH = Path("FINAL_10feature_XGB_for_deployment.pkl")
FINAL_CONFIG_PATH = Path("phase14B_final_robust_features.json")

# Optional full lexical model candidates.
# If none exists, the notebook will still complete the final-model analysis.
FULL_MODEL_CANDIDATES = [
    Path("FULL_14feature_XGB.pkl"),
    Path("full_lexical_xgb.pkl"),
    Path("FINAL_full_lexical_XGB.pkl"),
    Path("full_feature_xgb.pkl")
]

OUT = Path("experiment5_deployment_efficiency")
OUT.mkdir(exist_ok=True)

print("Final model:", FINAL_MODEL_PATH.resolve())
print("Config:", FINAL_CONFIG_PATH.resolve())
print("Data:", DATA_PATH.resolve())

In [ ]:
if not FINAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Final deployment model not found: {FINAL_MODEL_PATH.resolve()}"
    )

if not FINAL_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Final deployment config not found: {FINAL_CONFIG_PATH.resolve()}"
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Feature dataset not found: {DATA_PATH.resolve()}"
    )

final_model = joblib.load(
    FINAL_MODEL_PATH
)

with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    deployment_config = json.load(f)

FINAL_FEATURES = list(
    deployment_config["features"]
)

data = pd.read_csv(
    DATA_PATH
)

required = [
    "url",
    "label"
] + FINAL_FEATURES

missing = [
    c for c in required
    if c not in data.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("Dataset shape:", data.shape)
print("Final feature count:", len(FINAL_FEATURES))
print(FINAL_FEATURES)

In [ ]:
def normalize_url(url):
    url = str(url).strip()

    parse_url = url

    if not url.lower().startswith(
        ("http://", "https://")
    ):
        parse_url = "http://" + url

    parsed = urlparse(parse_url)

    hostname = (
        parsed.hostname.lower()
        if parsed.hostname
        else ""
    )

    if hostname.startswith("www."):
        hostname = hostname[4:]

    try:
        port = parsed.port
    except ValueError:
        port = None

    netloc = hostname

    if port is not None:
        netloc += f":{port}"

    scheme = parsed.scheme.lower()

    normalized = urlunparse(
        (
            scheme,
            netloc,
            parsed.path,
            "",
            parsed.query,
            ""
        )
    )

    return normalized


def ensure_scheme(url):

    url = str(url).strip()

    if not url:
        return ""

    has_real_scheme = re.match(
        r'^[A-Za-z][A-Za-z0-9+.\-]*://',
        url
    )

    if has_real_scheme:
        return url

    return "http://" + url


try:
    import tldextract

    domain_extractor = tldextract.TLDExtract(
        suffix_list_urls=None
    )

except Exception:
    domain_extractor = None


def get_registered_domain(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return ""

    if hostname == "fed.us" or hostname.endswith(".fed.us"):
        return "fed.us"

    if domain_extractor is None:
        return hostname

    ext = domain_extractor(hostname)

    if not ext.domain:
        return hostname

    if ext.suffix:
        return ext.domain + "." + ext.suffix

    return ext.domain


def count_subdomains(hostname):

    hostname = str(hostname).lower().strip(".")

    if not hostname:
        return 0

    registered_domain = get_registered_domain(
        hostname
    )

    hostname_parts = [
        x for x in hostname.split(".")
        if x
    ]

    domain_parts = [
        x for x in registered_domain.split(".")
        if x
    ]

    return max(
        len(hostname_parts)
        -
        len(domain_parts),
        0
    )


def parse_deployment_url(url):

    raw_url = str(url).strip()

    normalized_url = normalize_url(
        raw_url
    )

    try:
        raw_parsed = urlsplit(
            ensure_scheme(raw_url)
        )

        raw_hostname = (
            raw_parsed.hostname
            or ""
        ).lower()

    except Exception:
        raw_hostname = ""

    try:
        normalized_parsed = urlsplit(
            normalized_url
        )

        normalized_path = (
            normalized_parsed.path
            or ""
        )

        normalized_query = (
            normalized_parsed.query
            or ""
        )

    except Exception:
        normalized_path = ""
        normalized_query = ""

    return {
        "raw_url": raw_url,
        "normalized_url": normalized_url,
        "raw_hostname": raw_hostname,
        "path": normalized_path,
        "query": normalized_query
    }


SUSPICIOUS_KEYWORDS = [
    "login",
    "verify",
    "verification",
    "account",
    "secure",
    "update",
    "password",
    "signin",
    "confirm",
    "payment",
    "bank",
    "billing",
    "support"
]


def extract_deployment_features(url):

    parsed = parse_deployment_url(
        url
    )

    raw_url = parsed["raw_url"]
    raw_hostname = parsed["raw_hostname"]
    path = parsed["path"]
    query = parsed["query"]

    url_lower = raw_url.lower()

    url_length = len(raw_url)

    hostname_length = len(
        raw_hostname
    )

    path_length = len(path)
    query_length = len(query)

    num_digits = sum(
        c.isdigit()
        for c in raw_url
    )

    num_dots = raw_url.count(".")
    num_hyphens = raw_url.count("-")

    num_special_chars = sum(
        not c.isalnum()
        for c in raw_url
    )

    special_char_ratio = (
        num_special_chars
        /
        url_length
        if url_length > 0
        else 0.0
    )

    suspicious_keyword_count = sum(
        keyword in url_lower
        for keyword in SUSPICIOUS_KEYWORDS
    )

    num_subdomains = count_subdomains(
        raw_hostname
    )

    return {
        "num_subdomains":
            num_subdomains,

        "suspicious_keyword_count":
            suspicious_keyword_count,

        "num_digits":
            num_digits,

        "path_length":
            path_length,

        "hostname_length":
            hostname_length,

        "url_length":
            url_length,

        "num_dots":
            num_dots,

        "special_char_ratio":
            special_char_ratio,

        "num_hyphens":
            num_hyphens,

        "query_length":
            query_length
    }


assert list(
    extract_deployment_features(
        "https://www.example.com/a?x=1"
    ).keys()
) == FINAL_FEATURES

print("Verified 10-feature extractor loaded.")

In [ ]:
AUDIT_N = min(
    1000,
    len(data)
)

audit_df = data.sample(
    n=AUDIT_N,
    random_state=SEED
)

audit_records = []

for _, row in audit_df.iterrows():

    generated = extract_deployment_features(
        row["url"]
    )

    max_diff = max(
        abs(
            float(row[f])
            -
            float(generated[f])
        )
        for f in FINAL_FEATURES
    )

    audit_records.append(
        max_diff
    )

print("Audit samples:", AUDIT_N)
print("Maximum feature difference:", max(audit_records))
print("Mismatch count:", sum(x > 1e-12 for x in audit_records))

assert sum(x > 1e-12 for x in audit_records) == 0

In [ ]:
N_BENCHMARK = min(
    5000,
    len(data)
)

benchmark_df = (
    data
    .sample(
        n=N_BENCHMARK,
        random_state=SEED
    )
    .reset_index(drop=True)
)

benchmark_urls = (
    benchmark_df["url"]
    .astype(str)
    .tolist()
)

print("Benchmark samples:", N_BENCHMARK)

In [ ]:
final_model_size_bytes = (
    FINAL_MODEL_PATH.stat().st_size
)

final_model_size_kb = (
    final_model_size_bytes
    /
    1024
)

final_model_size_mb = (
    final_model_size_bytes
    /
    1024
    /
    1024
)

model_size_df = pd.DataFrame([
    {
        "model":
            "Final 10-feature XGBoost",

        "feature_count":
            len(FINAL_FEATURES),

        "size_bytes":
            final_model_size_bytes,

        "size_kb":
            final_model_size_kb,

        "size_mb":
            final_model_size_mb
    }
])

display(
    model_size_df.round(4)
)

In [ ]:
# Warm-up
for url in benchmark_urls[:200]:
    extract_deployment_features(url)

feature_times_ns = []

for url in benchmark_urls:

    t0 = time.perf_counter_ns()

    _ = extract_deployment_features(
        url
    )

    t1 = time.perf_counter_ns()

    feature_times_ns.append(
        t1 - t0
    )


feature_times_ms = (
    np.asarray(
        feature_times_ns,
        dtype=float
    )
    /
    1e6
)

feature_latency_summary = {
    "mean_ms":
        feature_times_ms.mean(),

    "median_ms":
        np.median(
            feature_times_ms
        ),

    "p95_ms":
        np.percentile(
            feature_times_ms,
            95
        ),

    "p99_ms":
        np.percentile(
            feature_times_ms,
            99
        )
}

print(feature_latency_summary)

In [ ]:
feature_records = [
    extract_deployment_features(
        url
    )
    for url in benchmark_urls
]

X_benchmark = pd.DataFrame(
    feature_records
)[FINAL_FEATURES]

print(X_benchmark.shape)

In [ ]:
# Warm-up
for i in range(
    min(
        200,
        len(X_benchmark)
    )
):

    _ = final_model.predict_proba(
        X_benchmark.iloc[
            [i]
        ]
    )


prediction_times_ns = []

for i in range(
    len(X_benchmark)
):

    x_one = X_benchmark.iloc[
        [i]
    ]

    t0 = time.perf_counter_ns()

    _ = final_model.predict_proba(
        x_one
    )

    t1 = time.perf_counter_ns()

    prediction_times_ns.append(
        t1 - t0
    )


prediction_times_ms = (
    np.asarray(
        prediction_times_ns,
        dtype=float
    )
    /
    1e6
)

prediction_latency_summary = {
    "mean_ms":
        prediction_times_ms.mean(),

    "median_ms":
        np.median(
            prediction_times_ms
        ),

    "p95_ms":
        np.percentile(
            prediction_times_ms,
            95
        ),

    "p99_ms":
        np.percentile(
            prediction_times_ms,
            99
        )
}

print(prediction_latency_summary)

In [ ]:
# Warm-up
for url in benchmark_urls[:200]:

    feat = extract_deployment_features(
        url
    )

    X_one = pd.DataFrame(
        [feat]
    )[FINAL_FEATURES]

    _ = final_model.predict_proba(
        X_one
    )


total_times_ns = []

for url in benchmark_urls:

    t0 = time.perf_counter_ns()

    feat = extract_deployment_features(
        url
    )

    X_one = pd.DataFrame(
        [feat]
    )[FINAL_FEATURES]

    _ = final_model.predict_proba(
        X_one
    )

    t1 = time.perf_counter_ns()

    total_times_ns.append(
        t1 - t0
    )


total_times_ms = (
    np.asarray(
        total_times_ns,
        dtype=float
    )
    /
    1e6
)

total_latency_summary = {
    "mean_ms":
        total_times_ms.mean(),

    "median_ms":
        np.median(
            total_times_ms
        ),

    "p95_ms":
        np.percentile(
            total_times_ms,
            95
        ),

    "p99_ms":
        np.percentile(
            total_times_ms,
            99
        )
}

print(total_latency_summary)

In [ ]:
BATCH_REPEATS = 30

# Warm-up
_ = final_model.predict_proba(
    X_benchmark
)

batch_times = []

for _ in range(
    BATCH_REPEATS
):

    t0 = time.perf_counter()

    _ = final_model.predict_proba(
        X_benchmark
    )

    t1 = time.perf_counter()

    batch_times.append(
        t1 - t0
    )


batch_times = np.asarray(
    batch_times
)

mean_batch_seconds = (
    batch_times.mean()
)

prediction_throughput = (
    len(
        X_benchmark
    )
    /
    mean_batch_seconds
)

print(
    "Mean batch inference time (s):",
    mean_batch_seconds
)

print(
    "Prediction throughput (URLs/s):",
    prediction_throughput
)

In [ ]:
summary_df = pd.DataFrame([
    {
        "component":
            "Feature extraction",

        "mean_ms":
            feature_latency_summary[
                "mean_ms"
            ],

        "median_ms":
            feature_latency_summary[
                "median_ms"
            ],

        "p95_ms":
            feature_latency_summary[
                "p95_ms"
            ],

        "p99_ms":
            feature_latency_summary[
                "p99_ms"
            ]
    },

    {
        "component":
            "Model prediction",

        "mean_ms":
            prediction_latency_summary[
                "mean_ms"
            ],

        "median_ms":
            prediction_latency_summary[
                "median_ms"
            ],

        "p95_ms":
            prediction_latency_summary[
                "p95_ms"
            ],

        "p99_ms":
            prediction_latency_summary[
                "p99_ms"
            ]
    },

    {
        "component":
            "Total URL-to-decision",

        "mean_ms":
            total_latency_summary[
                "mean_ms"
            ],

        "median_ms":
            total_latency_summary[
                "median_ms"
            ],

        "p95_ms":
            total_latency_summary[
                "p95_ms"
            ],

        "p99_ms":
            total_latency_summary[
                "p99_ms"
            ]
    }
])

display(
    summary_df.round(4)
)

In [ ]:
mean_total = (
    total_latency_summary[
        "mean_ms"
    ]
)

feature_share = (
    feature_latency_summary[
        "mean_ms"
    ]
    /
    mean_total
)

prediction_share = (
    prediction_latency_summary[
        "mean_ms"
    ]
    /
    mean_total
)

latency_contribution_df = pd.DataFrame([
    {
        "component":
            "Feature extraction",

        "mean_ms":
            feature_latency_summary[
                "mean_ms"
            ],

        "share_of_total":
            feature_share
    },

    {
        "component":
            "Prediction",

        "mean_ms":
            prediction_latency_summary[
                "mean_ms"
            ],

        "share_of_total":
            prediction_share
    }
])

display(
    latency_contribution_df.round(4)
)

In [ ]:
full_model_path = None

for candidate in FULL_MODEL_CANDIDATES:

    if candidate.exists():
        full_model_path = candidate
        break

if full_model_path is None:

    print(
        "No serialized full lexical model found."
    )

    print(
        "The final 10-feature deployment analysis is complete."
    )

else:

    print(
        "Found full lexical model:",
        full_model_path
    )

In [ ]:
comparison_df = None

if full_model_path is not None:

    full_model = joblib.load(
        full_model_path
    )

    full_size_bytes = (
        full_model_path
        .stat()
        .st_size
    )

    # Try common 14-feature lexical set:
    candidate_full_features = [
        c for c in data.columns
        if c in [
            "num_subdomains",
            "suspicious_keyword_count",
            "num_digits",
            "path_length",
            "hostname_length",
            "url_length",
            "num_dots",
            "special_char_ratio",
            "num_hyphens",
            "query_length",
            "num_special_chars",
            "has_ip",
            "has_https",
            "entropy"
        ]
    ]

    print(
        "Candidate full feature count:",
        len(candidate_full_features)
    )

    if (
        hasattr(
            full_model,
            "n_features_in_"
        )
        and
        full_model.n_features_in_
        !=
        len(candidate_full_features)
    ):

        print(
            "Full model feature count does not match the automatically inferred columns."
        )

        print(
            "Skipping latency comparison to avoid an invalid benchmark."
        )

    else:

        X_full = data.loc[
            benchmark_df.index,
            candidate_full_features
        ].copy()

        _ = full_model.predict_proba(
            X_full
        )

        full_batch_times = []

        for _ in range(
            BATCH_REPEATS
        ):

            t0 = time.perf_counter()

            _ = full_model.predict_proba(
                X_full
            )

            t1 = time.perf_counter()

            full_batch_times.append(
                t1 - t0
            )

        full_batch_times = np.asarray(
            full_batch_times
        )

        full_throughput = (
            len(X_full)
            /
            full_batch_times.mean()
        )

        comparison_df = pd.DataFrame([
            {
                "model":
                    "Final 10-feature",

                "feature_count":
                    len(FINAL_FEATURES),

                "model_size_kb":
                    final_model_size_kb,

                "batch_throughput_urls_per_s":
                    prediction_throughput
            },

            {
                "model":
                    "Full lexical",

                "feature_count":
                    len(candidate_full_features),

                "model_size_kb":
                    full_size_bytes
                    /
                    1024,

                "batch_throughput_urls_per_s":
                    full_throughput
            }
        ])

        display(
            comparison_df.round(4)
        )

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.bar(
    summary_df[
        "component"
    ],
    summary_df[
        "mean_ms"
    ]
)

ax.set_ylabel(
    "Mean latency (ms)"
)

ax.set_title(
    "Deployment Latency by Pipeline Component"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

In [ ]:
model_size_df.to_csv(
    OUT / "model_size_summary.csv",
    index=False
)

summary_df.to_csv(
    OUT / "latency_summary.csv",
    index=False
)

latency_contribution_df.to_csv(
    OUT / "latency_contribution.csv",
    index=False
)

pd.DataFrame({
    "feature_extraction_ms":
        feature_times_ms,

    "prediction_ms":
        prediction_times_ms,

    "total_url_to_decision_ms":
        total_times_ms
}).to_csv(
    OUT / "sample_level_latency.csv",
    index=False
)

throughput_df = pd.DataFrame([
    {
        "n_samples":
            len(X_benchmark),

        "batch_repeats":
            BATCH_REPEATS,

        "mean_batch_seconds":
            mean_batch_seconds,

        "prediction_throughput_urls_per_s":
            prediction_throughput
    }
])

throughput_df.to_csv(
    OUT / "throughput_summary.csv",
    index=False
)

if comparison_df is not None:

    comparison_df.to_csv(
        OUT / "full_vs_lightweight_efficiency.csv",
        index=False
    )

print(
    "Saved outputs to:",
    OUT.resolve()
)

for p in OUT.iterdir():
    print("-", p.name)

## Interpretation

The final 10-feature XGBoost detector demonstrated a small computational footprint. The serialized model occupied approximately 0.58 MB, while mean lexical feature extraction required only 0.085 ms per URL. Model inference required approximately 2.22 ms, and the complete URL-to-decision pipeline averaged 3.75 ms, with a 95th-percentile latency of approximately 5.96 ms.

Feature extraction accounted for only a small proportion of the total processing time, indicating that the compact lexical representation introduces negligible computational overhead. Batch inference also achieved high throughput on the evaluation machine.

These results support the deployment value of the final 10-feature representation. Together with the previously established non-inferiority of the reduced feature set, the detector preserves strong predictive performance while maintaining a small model size and low decision latency. Combined with Experiment 4, the results further suggest that the principal practical bottleneck of the complete system lies in QR payload extraction rather than downstream lexical feature processing or classification.